# Previsão de Dengue nos Municípios de São Paulo com Clima, Rt e Limiares Epidêmicos

**Autores:** _Nome Sobrenome_, _Nome Sobrenome_, _Nome Sobrenome_, _Nome Sobrenome_, _Nome Sobrenome_

**Curso:** _nome do curso — instituição_ · **Data:** _mês/ano_

---

## 1. Introdução

A **dengue** é a arbovirose urbana de maior impacto no Brasil. Sua transmissão depende do mosquito *Aedes aegypti*, cujo ciclo de vida é sensível à **temperatura** e à **umidade**: em períodos quentes e úmidos o mosquito se desenvolve mais rápido, o período de incubação do vírus no vetor encurta e a transmissão aumenta. Por isso a doença tem forte sazonalidade, com picos entre fevereiro e maio na Região Sudeste.

O [InfoDengue](https://info.dengue.mat.br/informacoes/), sistema de alerta desenvolvido pela Fiocruz e pela FGV, combina notificações de casos, clima e indicadores epidemiológicos — como o **número reprodutivo efetivo (Rt)** e **limiares epidêmicos** calculados para cada município — para acompanhar a situação semanal de cada cidade brasileira.

Este projeto adota a mesma lógica, mas constrói tudo **a partir das fontes primárias**, sem usar APIs intermediárias.

### Objetivos

1. Montar uma base semanal para os **645 municípios do estado de São Paulo** (2014–2025), consultando diretamente o SINAN, o Copernicus (ERA5-Land) e o IBGE.
2. Estimar o **Rt** e os **limiares epidêmicos** (pré-epidêmico, pós-epidêmico e epidêmico) de cada município.
3. Treinar modelos de aprendizado de máquina para:
   - **regressão**: prever a incidência de dengue 1 e 4 semanas à frente;
   - **classificação**: prever se a incidência vai **ultrapassar o limiar epidêmico** do município.
4. Avaliar os modelos com **validação temporal em janelas móveis** (re-treinando a cada ano), com previsões pontuais e com faixas de incerteza, e comparar com previsões de referência (*baselines*).

### Perguntas de pesquisa

1. É possível prever, com 1 e 4 semanas de antecedência, **quantos casos de dengue** cada município de SP terá?
2. É possível antecipar **quais municípios vão entrar em situação epidêmica**?
3. Quanto o **clima** (temperatura e umidade) e a situação dos **municípios vizinhos** ajudam nessas previsões?
4. Que **ações** de vigilância e controle essas previsões permitem orientar, e com quais riscos?

### Variáveis-alvo

| Tarefa | Variável-alvo | Tipo | Definição |
|---|---|---|---|
| Regressão | `y` — incidência na semana `T + h` | contínua (casos por 100 mil hab.) | notificações de dengue de residentes do município, por semana de início dos sintomas, ÷ população × 100 mil |
| Classificação | `evento` — semana epidêmica em `T + h` | binária (0/1) | 1 se a incidência passar do **limiar epidêmico** do município (seção 8) **e** houver mais de 10 casos; 0 caso contrário |

`T` é a semana em que a previsão é feita e `h` (1 ou 4 semanas) é o horizonte.

<small>*Nota*: previsões de doenças devem ser usadas junto com a vigilância epidemiológica, o monitoramento de vetores e a avaliação de especialistas.</small>

## 2. Dados

### 2.1 Fontes

| Fonte | Conteúdo | Forma de acesso |
|---|---|---|
| **SINAN** — Ministério da Saúde | Notificações individuais de dengue, um arquivo nacional por ano (`DENGBRaa`) | Portal de Dados Abertos do SUS (arquivos Parquet/CSV) |
| **ERA5-Land** — Copernicus/ECMWF | Reanálise climática com resolução de 0,1° (~9 km): temperatura e ponto de orvalho a 2 m, agregados diários | Google Earth Engine, coleção `ECMWF/ERA5_LAND/DAILY_AGGR` |
| **IBGE** — SIDRA | Estimativas de população (tabela 6579) e Censo 2022 (tabela 4709) | API SIDRA |
| **IBGE** — Malhas e Localidades | Polígonos e nomes dos municípios | APIs de serviços de dados |

### 2.2 Variáveis construídas (município × semana epidemiológica)

| Variável | Descrição |
|---|---|
| `casos` | notificações de dengue por **semana de início dos sintomas**, município de residência |
| `atraso` | semanas entre o início dos sintomas e a digitação no SINAN |
| `pop` | população residente |
| `incidencia` | casos por 100 mil habitantes |
| `temp_min`, `temp_med`, `temp_max` | médias semanais das temperaturas diárias mínima, média e máxima (°C) |
| `umid_min`, `umid_med`, `umid_max` | umidade relativa do ar (%) |
| `Rt`, `p1` | número reprodutivo efetivo e probabilidade de Rt > 1 |
| `limiar_pre`, `limiar_pos`, `limiar_epidemico` | limiares de incidência do município |

### 2.3 Como executar

- O notebook foi feito para o **Google Colab** e também roda em Jupyter local.
- **Clima:** é preciso acesso gratuito ao [Google Earth Engine](https://code.earthengine.google.com/register) para uso não comercial (acadêmico), com a mesma conta Google do Colab. No cadastro é criado um projeto do Google Cloud; coloque o ID desse projeto em `GEE_PROJETO`, no início da célula de clima (seção 4.5). Na primeira execução o Colab pede autorização de acesso ao Earth Engine.
- As exportações do Earth Engine levam de alguns minutos a cerca de uma hora e podem ser acompanhadas em <https://code.earthengine.google.com/tasks>. **Tudo fica em cache**: se o arquivo já existe, não é baixado nem exportado de novo, então é possível interromper e retomar.
- Para economizar espaço, cada arquivo anual do SINAN é baixado no disco temporário do Colab, filtrado para SP, agregado e apagado. No Google Drive ficam os dados já processados (dezenas de MB) e as tabelas diárias de clima exportadas pelo Earth Engine (~200 MB, que podem ser apagadas depois). As pastas são criadas automaticamente.

In [ ]:
# Dependências (o Colab já traz a maioria)
%pip install -q earthengine-api shapely pyarrow scipy scikit-learn requests

## 3. Configuração

In [ ]:
import json, os, shutil, sys, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

from scipy import stats
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, precision_score,
                             recall_score, f1_score, average_precision_score, precision_recall_curve)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.simplefilter("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
# ---------------- Parâmetros do estudo ----------------
UF_CODIGO, UF_SIGLA = "35", "SP"
ANO_INICIO, ANO_FIM = 2014, 2025
ANOS_SINAN = range(ANO_INICIO, ANO_FIM + 2)   # inclui o arquivo do ano seguinte (sintomas em dezembro, notificação em janeiro)
ANOS_CLIMA = range(ANO_INICIO - 1, ANO_FIM + 1)  # um ano antes, para as defasagens climáticas

COL_MUNICIPIO = "ID_MN_RESI"   # município de residência
MAX_ATRASO = 26                # atrasos maiores são agrupados nesta classe (semanas)

TREINO_ATE = pd.Timestamp("2021-12-31")   # limiares epidêmicos e projeção sazonal usam dados até 2021
                                          # (os modelos de previsão usam janelas móveis: seção 9.4)
HORIZONTES = [1, 4]                        # semanas à frente
RETREINAR_MODELOS = False                  # False: reaproveita os resultados dos modelos salvos no Drive
                                           # (mude para True se alterar variáveis, modelos ou período)
RANDOM_STATE = 42

# Pasta de dados: no Colab, o Google Drive guarda o cache entre sessões
NO_COLAB = "google.colab" in sys.modules
USAR_GOOGLE_DRIVE = True
if NO_COLAB and USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_DADOS = Path("/content/drive/MyDrive/dengue_sp/dados")
    DIR_TEMP = Path("/content/temp_sinan")     # disco local do Colab: arquivos nacionais grandes e temporários
else:
    DIR_DADOS = Path("dados")
    DIR_TEMP = DIR_DADOS / "bruto"
DIR_BRUTO = DIR_DADOS / "bruto"
for d in (DIR_BRUTO / "ibge", DIR_BRUTO / "era5", DIR_DADOS / "sinan", DIR_TEMP):
    d.mkdir(parents=True, exist_ok=True)   # as pastas são criadas automaticamente
print("Dados em:", DIR_DADOS.resolve())

### 3.1 Funções auxiliares

A **semana epidemiológica (SE)** começa no domingo; a SE 1 é a primeira semana do ano com pelo menos quatro dias em janeiro. Para evitar problemas com anos de 53 semanas, o tempo é indexado pela **data do domingo** que inicia cada semana, e o par (ano, SE) é calculado a partir dela.

In [ ]:
def inicio_semana(datas):
    """Domingo que inicia a semana epidemiológica de cada data."""
    d = pd.to_datetime(pd.Series(datas)).dt.normalize()
    return d - pd.to_timedelta((d.dt.weekday + 1) % 7, unit="D")


def ano_se(semanas):
    """(ano epidemiológico, SE) a partir do domingo de início: a quarta-feira define o ano."""
    qua = pd.to_datetime(pd.Series(semanas)) + pd.Timedelta(days=3)
    return qua.dt.year.to_numpy(), ((qua.dt.dayofyear - 1) // 7 + 1).to_numpy()


def para_data(s):
    """Converte datas nos formatos AAAA-MM-DD, AAAAMMDD ou DD/MM/AAAA, tolerando valores inválidos."""
    s = pd.Series(s, dtype="object").astype(str).str.strip().str[:10]
    d = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    for fmt, padrao in (("%Y%m%d", r"\d{8}"), ("%d/%m/%Y", r"\d{2}/\d{2}/\d{4}")):
        m = d.isna() & s.str.fullmatch(padrao)
        if m.any():
            d[m] = pd.to_datetime(s[m], format=fmt, errors="coerce")
    return d


def baixar(url, destino, params=None, tentativas=3):
    """Baixa um arquivo em blocos, com novas tentativas; se ele já existe, não baixa de novo."""
    import time
    destino = Path(destino)
    if destino.exists():
        return destino
    tmp = destino.with_name(destino.name + ".part")
    for tentativa in range(1, tentativas + 1):
        try:
            print(f"Baixando {destino.name} ...", end=" ", flush=True)
            with requests.get(url, params=params, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for bloco in r.iter_content(chunk_size=1 << 20):
                        f.write(bloco)
            tmp.replace(destino)
            print(f"{destino.stat().st_size / 1e6:.1f} MB")
            return destino
        except requests.HTTPError:
            print("não encontrado")
            raise
        except requests.RequestException as e:
            print(f"falhou (tentativa {tentativa}/{tentativas}): {type(e).__name__}")
            if tentativa == tentativas:
                raise
            time.sleep(10 * tentativa)


_t = inicio_semana(["2024-01-01", "2021-01-01"])
print("Conferência:", [(str(d.date()), int(a), int(s)) for d, a, s in zip(_t, *ano_se(_t))],
      "-> esperado SE 1/2024 e SE 53/2020")

## 4. Extração dos dados

### 4.1 Arquivos do IBGE

São quatro arquivos pequenos: lista de municípios, malha territorial (polígonos) e duas tabelas de população do SIDRA.

O servidor do IBGE às vezes fica instável ou não responde a servidores fora do Brasil, como os do Colab. Se o download falhar mesmo depois das novas tentativas, a célula mostra os links: basta abri-los no navegador do seu computador, salvar cada arquivo **com o nome indicado** na pasta indicada do Google Drive e rodar a célula de novo.

In [ ]:
DIR_IBGE = DIR_BRUTO / "ibge"
# anos com estimativa na tabela 6579 (não há 2022 e 2023; 2022 vem do Censo e 2023 é interpolado)
ANOS_ESTIMATIVAS = ",".join(str(a) for a in range(ANO_INICIO - 1, ANO_FIM + 1) if a not in (2022, 2023))
ARQUIVOS_IBGE = {
    f"municipios_{UF_SIGLA}.json": (f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_CODIGO}/municipios", None),
    f"malha_{UF_SIGLA}.geojson": (f"https://servicodados.ibge.gov.br/api/v3/malhas/estados/{UF_CODIGO}",
                                  {"formato": "application/vnd.geo+json", "intrarregiao": "municipio", "qualidade": "intermediaria"}),
    f"pop_estimativas_{UF_SIGLA}.json": (f"https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%20{UF_CODIGO}/v/9324/p/{ANOS_ESTIMATIVAS}", {"formato": "json"}),
    f"pop_censo2022_{UF_SIGLA}.json": (f"https://apisidra.ibge.gov.br/values/t/4709/n6/in%20n3%20{UF_CODIGO}/v/93/p/2022", {"formato": "json"}),
}

faltando = []
for nome, (url, params) in ARQUIVOS_IBGE.items():
    try:
        baixar(url, DIR_IBGE / nome, params=params)
    except requests.RequestException:
        faltando.append((nome, requests.Request("GET", url, params=params).prepare().url))

if faltando:
    pasta = "MyDrive/dengue_sp/dados/bruto/ibge" if (NO_COLAB and USAR_GOOGLE_DRIVE) else str(DIR_IBGE.resolve())
    print("\n" + "=" * 90)
    print("O IBGE não respondeu. Abra cada link abaixo no navegador do seu computador, salve a página")
    print(f"(Ctrl+S) com o nome indicado e coloque o arquivo na pasta: {pasta}\n")
    for nome, link in faltando:
        print(f"  nome do arquivo: {nome}\n  link: {link}\n")
    print("Depois rode esta célula de novo.")
    print("=" * 90)
    raise RuntimeError(f"{len(faltando)} arquivo(s) do IBGE pendente(s) — veja as instruções acima.")
print("Arquivos do IBGE prontos.")

### 4.2 Municípios e malha territorial

In [ ]:
from shapely.geometry import shape

arq_mun, arq_malha = DIR_IBGE / f"municipios_{UF_SIGLA}.json", DIR_IBGE / f"malha_{UF_SIGLA}.geojson"
geojson = json.loads(arq_malha.read_text(encoding="utf-8-sig"))
geoms = {int(f["properties"]["codarea"]): shape(f["geometry"]) for f in geojson["features"]}
nomes = {int(m["id"]): m["nome"] for m in json.loads(arq_mun.read_text(encoding="utf-8-sig"))}

muns = pd.DataFrame({"geocodigo": sorted(geoms)})
muns["geocod6"] = muns.geocodigo.astype(str).str[:6]      # o SINAN usa o código de 6 dígitos
muns["nome"] = muns.geocodigo.map(nomes)
M = len(muns)
print(f"{M} municípios")
muns.head()

### 4.3 População (IBGE/SIDRA)

A tabela 6579 traz as estimativas anuais; a tabela 4709, o Censo 2022. Anos sem dado são interpolados linearmente para cada município.

In [ ]:
def ler_sidra(arquivo):
    dados = json.loads(Path(arquivo).read_text(encoding="utf-8-sig"))
    cab, linhas = dados[0], pd.DataFrame(dados[1:])
    col_mun = next(k for k, v in cab.items() if v.startswith("Município") and v.endswith("(Código)"))
    col_ano = next(k for k, v in cab.items() if v.startswith("Ano") and v.endswith("(Código)"))
    return pd.DataFrame({"geocodigo": linhas[col_mun].astype(int), "ano": linhas[col_ano].astype(int),
                         "pop": pd.to_numeric(linhas["V"], errors="coerce")}).dropna()


pop_long = pd.concat([ler_sidra(DIR_IBGE / f"pop_estimativas_{UF_SIGLA}.json"),
                      ler_sidra(DIR_IBGE / f"pop_censo2022_{UF_SIGLA}.json")])
pop_long = pop_long.drop_duplicates(["geocodigo", "ano"], keep="last")
pop_anual = (pop_long.pivot(index="ano", columns="geocodigo", values="pop")
                     .reindex(index=range(ANO_INICIO - 1, ANO_FIM + 2), columns=muns.geocodigo)
                     .interpolate(limit_direction="both"))
print("População do estado (milhões):")
print((pop_anual.loc[ANO_INICIO:ANO_FIM].sum(axis=1) / 1e6).round(2).to_string())

### 4.4 Notificações de dengue (SINAN)

Cada arquivo anual tem todas as notificações do país. O processamento lê **apenas as colunas necessárias**, em blocos, mantém os residentes em SP e agrega em **município × semana de início dos sintomas × atraso de digitação**.

| Coluna | Uso |
|---|---|
| `DT_SIN_PRI` | data de início dos sintomas (se vazia, usa `DT_NOTIFIC`) |
| `DT_NOTIFIC` | data da notificação |
| `DT_DIGITA` | data de digitação no sistema → atraso |
| `ID_MN_RESI` | município de residência (6 dígitos) |

Guardar o **atraso** permite saber, para qualquer semana do passado, quantos casos **já estavam registrados** naquele momento. Isso é usado na modelagem para que as variáveis reflitam o que realmente se sabia na data da previsão.

In [ ]:
URL_SINAN = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/{fmt}/DENGBR{aa}.{fmt}.zip"
COLS_SINAN = ["DT_SIN_PRI", "DT_NOTIFIC", "DT_DIGITA", "ID_MN_RESI"]
TAM_BLOCO = 500_000


def agrega_bloco(df):
    """Mantém residentes na UF e conta notificações por (município, semana dos sintomas, atraso)."""
    df = df.rename(columns=lambda c: c.strip().upper())
    mun = df[COL_MUNICIPIO].astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str[:6]
    manter = mun.str.startswith(UF_CODIGO)
    df, mun = df[manter], mun[manter]
    if df.empty:
        return None
    dt_sin = para_data(df["DT_SIN_PRI"]) if "DT_SIN_PRI" in df else pd.Series(pd.NaT, index=df.index)
    if "DT_NOTIFIC" in df:
        dt_sin = dt_sin.fillna(para_data(df["DT_NOTIFIC"]))
    semana = inicio_semana(dt_sin).set_axis(df.index)
    if "DT_DIGITA" in df:
        atraso = ((inicio_semana(para_data(df["DT_DIGITA"])).set_axis(df.index) - semana).dt.days // 7)
        atraso = atraso.fillna(0).clip(0, MAX_ATRASO)
    else:
        atraso = pd.Series(0, index=df.index)
    out = pd.DataFrame({"geocod6": mun.values, "semana": semana.values, "atraso": atraso.astype(int).values})
    return out.dropna(subset=["semana"]).groupby(["geocod6", "semana", "atraso"]).size().rename("n").reset_index()


def blocos_parquet(arq_zip, pasta_tmp):
    import pyarrow.parquet as pq
    with zipfile.ZipFile(arq_zip) as z:
        membros = [m for m in z.namelist() if m.lower().endswith(".parquet")]
        z.extractall(pasta_tmp, members=membros)
    for m in membros:
        pf = pq.ParquetFile(Path(pasta_tmp) / m)
        nomes_col = {c.upper(): c for c in pf.schema_arrow.names}
        cols = [nomes_col[c] for c in COLS_SINAN if c in nomes_col]
        for lote in pf.iter_batches(batch_size=TAM_BLOCO, columns=cols):
            yield lote.to_pandas().astype(str).replace({"None": None, "nan": None, "NaT": None})


def blocos_csv(arq_zip):
    with zipfile.ZipFile(arq_zip) as z:
        membro = next(m for m in z.namelist() if m.lower().endswith(".csv"))
        with z.open(membro) as f:
            cabecalho = f.readline().decode("latin-1")
        sep = max([",", ";", "|", "\t"], key=cabecalho.count)
        with z.open(membro) as f:
            yield from pd.read_csv(f, sep=sep, dtype=str, encoding="latin-1", chunksize=TAM_BLOCO,
                                   usecols=lambda c: c.strip().upper() in COLS_SINAN)


def processar_sinan(ano):
    aa = f"{ano % 100:02d}"
    destino = DIR_DADOS / "sinan" / f"DENG{aa}_{UF_SIGLA}.parquet"
    if destino.exists():
        return destino
    for fmt in ("parquet", "csv"):          # Parquet é menor; CSV é a alternativa
        arq = DIR_TEMP / f"DENGBR{aa}.{fmt}.zip"
        try:
            baixar(URL_SINAN.format(fmt=fmt, aa=aa), arq)
            break
        except requests.RequestException as e:
            print(f"{arq.name} indisponível ({e}); tentando outro formato")
    else:
        raise RuntimeError(f"SINAN {ano} não encontrado")
    pasta_tmp = DIR_TEMP / f"tmp_{aa}"
    blocos = blocos_parquet(arq, pasta_tmp) if fmt == "parquet" else blocos_csv(arq)
    partes = [p for p in map(agrega_bloco, blocos) if p is not None]
    agreg = pd.concat(partes).groupby(["geocod6", "semana", "atraso"])["n"].sum().reset_index()
    agreg.to_parquet(destino, index=False)
    shutil.rmtree(pasta_tmp, ignore_errors=True)
    arq.unlink(missing_ok=True)              # apaga o arquivo nacional
    print(f"SINAN {ano}: {agreg.n.sum():,} notificações de residentes em {UF_SIGLA}")
    return destino


sinan = pd.concat([pd.read_parquet(processar_sinan(a)) for a in ANOS_SINAN])
sinan["semana"] = pd.to_datetime(sinan["semana"])
sinan = sinan.groupby(["geocod6", "semana", "atraso"])["n"].sum().reset_index()
print(f"\nTotal: {sinan.n.sum():,} notificações | {sinan.geocod6.nunique()} municípios com casos")

### 4.5 Clima (ERA5-Land, via Google Earth Engine)

O clima vem do **ERA5-Land**, a reanálise do Copernicus/ECMWF com resolução de 0,1° (~9 km), também usada pelo InfoDengue. O acesso é pelo **Google Earth Engine**, que mantém o ERA5-Land com os agregados diários já calculados (`ECMWF/ERA5_LAND/DAILY_AGGR`). Assim não há a fila de processamento do Climate Data Store do Copernicus.

| Variável do ERA5-Land | Variável resultante |
|---|---|
| temperatura a 2 m — mínimo diário | `temp_min` |
| temperatura a 2 m — média diária | `temp_med` |
| temperatura a 2 m — máximo diário | `temp_max` |
| ponto de orvalho a 2 m — média diária | umidade relativa |

Para cada ano, o Earth Engine calcula a **média diária de cada município**, ponderada pela fração de cada pixel dentro do polígono (assim municípios menores que um pixel também recebem valor), e exporta uma tabela CSV para o Google Drive. As 13 exportações (2013 a 2025) rodam em paralelo nos servidores do Google.

A umidade relativa vem da fórmula de Magnus, $UR = 100\, e_s(T_d)/e_s(T)$, com $e_s(T) = 6{,}112 \exp\!\left(\frac{17{,}62\,T}{243{,}12+T}\right)$. A umidade média usa a temperatura média; a máxima usa a temperatura mínima e a mínima usa a máxima (aproximação, já que o ponto de orvalho varia pouco ao longo do dia). Por fim, os valores diários são agregados em médias semanais.

<small>Os dias do agregado diário seguem o horário UTC (3 horas à frente de Brasília); o efeito sobre médias semanais é desprezível.</small>

In [ ]:
import time
import ee
from shapely.geometry import mapping

GEE_PROJETO = "COLE-AQUI-O-ID-DO-PROJETO"   # ID do projeto do Google Cloud registrado no Earth Engine
COLECAO_GEE = "ECMWF/ERA5_LAND/DAILY_AGGR"
BANDAS_GEE = {"temperature_2m_min": "tmin", "temperature_2m": "tmed",
              "temperature_2m_max": "tmax", "dewpoint_temperature_2m": "tdmed"}
PASTA_DRIVE_GEE = "dengue_sp_gee"            # pasta criada pelo Earth Engine na raiz do Google Drive
DIR_GEE = Path("/content/drive/MyDrive") / PASTA_DRIVE_GEE if NO_COLAB else DIR_DADOS / "gee"


def nome_exportacao(ano):
    return f"clima_{UF_SIGLA}_era5land_{ano}"


def municipios_ee():
    """Polígonos dos municípios como FeatureCollection do Earth Engine (simplificados ~500 m, bem menor que um pixel)."""
    feats = [ee.Feature(ee.Geometry(mapping(geoms[g].simplify(0.005, preserve_topology=True))), {"geocodigo": int(g)})
             for g in muns.geocodigo]
    return ee.FeatureCollection(feats)


def exportar_ano(ano, fc):
    """Média diária de cada município (média ponderada pela área de cada pixel) -> CSV no Google Drive."""
    col = (ee.ImageCollection(COLECAO_GEE).filterDate(f"{ano}-01-01", f"{ano + 1}-01-01").select(list(BANDAS_GEE)))

    def por_dia(img):
        data = img.date().format("YYYY-MM-dd")
        return img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=11132).map(lambda f: f.set("data", data))

    tarefa = ee.batch.Export.table.toDrive(
        collection=col.map(por_dia).flatten(), description=nome_exportacao(ano), folder=PASTA_DRIVE_GEE,
        fileNamePrefix=nome_exportacao(ano), fileFormat="CSV", selectors=["geocodigo", "data"] + list(BANDAS_GEE))
    tarefa.start()
    return tarefa


def umidade_relativa(t, td):
    es = lambda x: 6.112 * np.exp(17.62 * x / (243.12 + x))
    return np.clip(100 * es(td) / es(t), 0, 100)


arq_clima = DIR_DADOS / f"clima_{UF_SIGLA}_semanal.parquet"
if arq_clima.exists():
    clima = pd.read_parquet(arq_clima)
else:
    DIR_GEE.mkdir(parents=True, exist_ok=True)
    faltando = [a for a in ANOS_CLIMA if not list(DIR_GEE.glob(f"{nome_exportacao(a)}*.csv"))]

    # 1) exportações no Earth Engine (uma tarefa por ano, rodando em paralelo nos servidores do Google)
    if faltando:
        if not NO_COLAB:
            raise RuntimeError(f"Fora do Colab: baixe do Google Drive (pasta {PASTA_DRIVE_GEE}) os CSV "
                               f"{[nome_exportacao(a) + '.csv' for a in faltando]} para {DIR_GEE.resolve()}")
        if GEE_PROJETO.startswith("COLE-AQUI"):
            raise RuntimeError("Preencha GEE_PROJETO (no início desta célula) com o ID do seu projeto do Earth Engine.")
        if not Path("/content/drive/MyDrive").exists():
            from google.colab import drive
            drive.mount("/content/drive")
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJETO)
        try:
            ativas = {t.config.get("description"): t for t in ee.batch.Task.list() if str(t.state) in ("READY", "RUNNING")}
        except Exception:
            ativas = {}
        fc = municipios_ee()
        tarefas = {}
        for ano in faltando:
            nome = nome_exportacao(ano)
            tarefas[ano] = ativas.get(nome) or exportar_ano(ano, fc)   # não reenvia o que já está rodando
        print(f"{len(tarefas)} exportações no Earth Engine (acompanhe em https://code.earthengine.google.com/tasks)")
        estados_anteriores = {}
        while True:
            estados = {ano: t.status() for ano, t in tarefas.items()}
            for ano, st in estados.items():
                if st["state"] != estados_anteriores.get(ano):
                    print(f"  {ano}: {st['state']}" + (f" — {st.get('error_message', '')}" if st["state"] == "FAILED" else ""), flush=True)
            estados_anteriores = {ano: st["state"] for ano, st in estados.items()}
            if all(st["state"] in ("COMPLETED", "FAILED", "CANCELLED") for st in estados.values()):
                break
            time.sleep(30)
        falhas = [a for a, st in estados.items() if st["state"] != "COMPLETED"]
        if falhas:
            raise RuntimeError(f"Exportações sem sucesso: {falhas}. Rode a célula de novo para tentar só esses anos.")
        # o Drive montado no Colab pode levar alguns minutos para mostrar os arquivos novos
        for _ in range(40):
            if all(list(DIR_GEE.glob(f"{nome_exportacao(a)}*.csv")) for a in faltando):
                break
            time.sleep(15)
        else:
            raise RuntimeError(f"As exportações terminaram, mas os CSV ainda não apareceram em {DIR_GEE}. "
                               "Espere alguns minutos e rode a célula de novo.")

    # 2) processamento: diário -> semanal, por município
    diario = pd.concat([pd.read_csv(f) for a in ANOS_CLIMA for f in sorted(DIR_GEE.glob(f"{nome_exportacao(a)}*.csv"))])
    diario = diario.rename(columns=BANDAS_GEE)
    for c in BANDAS_GEE.values():
        diario[c] = diario[c] - 273.15                       # Kelvin -> Celsius
    diario["temp_min"], diario["temp_med"], diario["temp_max"] = diario.tmin, diario.tmed, diario.tmax
    diario["umid_med"] = umidade_relativa(diario.tmed, diario.tdmed)
    diario["umid_max"] = umidade_relativa(diario.tmin, diario.tdmed)
    diario["umid_min"] = umidade_relativa(diario.tmax, diario.tdmed)
    diario["semana"] = inicio_semana(diario["data"]).to_numpy()
    vars_clima = ["temp_min", "temp_med", "temp_max", "umid_min", "umid_med", "umid_max"]
    clima = diario.groupby(["semana", "geocodigo"])[vars_clima].mean().reset_index()
    # município sem valor numa semana (não deve acontecer) -> média do estado naquela semana
    clima[vars_clima] = clima[vars_clima].fillna(clima.groupby("semana")[vars_clima].transform("mean"))
    clima.to_parquet(arq_clima, index=False)
    print(f"Clima semanal salvo em {arq_clima} ({len(clima):,} linhas). "
          f"Os CSV diários em {PASTA_DRIVE_GEE} podem ser apagados, se quiser liberar espaço.")
clima["semana"] = pd.to_datetime(clima["semana"])
clima.drop(columns=["semana", "geocodigo"]).describe().round(1).T[["mean", "min", "max"]]

## 5. Base município × semana

Os dados são organizados em **matrizes de semanas × municípios**, formato que torna rápidos os cálculos por município (Rt, limiares, defasagens).

`C[t, m, k]` guarda quantos casos com sintomas na semana `t`, no município `m`, já estavam digitados **até `k` semanas depois**. A última camada (`k = 26`) é o total de casos.

In [ ]:
semanas = pd.date_range(inicio_semana([f"{ANO_INICIO}-01-04"])[0], inicio_semana([f"{ANO_FIM}-12-28"])[0], freq="7D")
ANO_EPI, SE = ano_se(semanas)
semanas, ANO_EPI, SE = semanas[ANO_EPI <= ANO_FIM], ANO_EPI[ANO_EPI <= ANO_FIM], SE[ANO_EPI <= ANO_FIM]
T = len(semanas)
TEMPORADA = np.where(SE >= 41, ANO_EPI, ANO_EPI - 1)    # temporada de dengue: SE 41 a SE 40 do ano seguinte

t_idx = semanas.get_indexer(sinan.semana)
m_idx = pd.Index(muns.geocod6).get_indexer(sinan.geocod6)
ok = (t_idx >= 0) & (m_idx >= 0)
C = np.zeros((T, M, MAX_ATRASO + 1), dtype=np.int32)
np.add.at(C, (t_idx[ok], m_idx[ok], sinan.atraso.to_numpy()[ok]), sinan.n.to_numpy()[ok])
C = C.cumsum(axis=2)

CASOS = C[:, :, -1].astype(float)
POP = pop_anual.reindex(ANO_EPI).to_numpy()
INC = CASOS / POP * 1e5                                  # incidência semanal por 100 mil hab.
VARS_CLIMA = ["temp_min", "temp_med", "temp_max", "umid_min", "umid_med", "umid_max"]
CLIMA = {v: clima.pivot(index="semana", columns="geocodigo", values=v)
              .reindex(index=semanas, columns=muns.geocodigo).to_numpy() for v in VARS_CLIMA}
CAPITAL = int(np.where(muns.geocodigo == 3550308)[0][0]) if (muns.geocodigo == 3550308).any() else int(np.argmax(POP[-1]))

print(f"{T} semanas ({semanas[0].date()} a {semanas[-1].date()}) × {M} municípios")
print(f"Casos no período: {CASOS.sum():,.0f} | fora do período ou sem município válido: {sinan.n[~ok].sum():,}")
print(f"Semanas-município sem clima: {np.isnan(CLIMA['temp_med']).mean():.1%}")

## 6. Análise exploratória

### 6.1 Série histórica do estado

In [ ]:
inc_estado = CASOS.sum(axis=1) / POP.sum(axis=1) * 1e5
fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.fill_between(semanas, inc_estado, color="#c0392b", alpha=0.7, label="incidência")
ax1.set_ylabel("casos / 100 mil hab. por semana")
ax2 = ax1.twinx()
ax2.plot(semanas, np.nanmean(CLIMA["temp_min"], axis=1), color="#2c3e50", lw=0.8, label="temp. mínima")
ax2.set_ylabel("temperatura mínima média (°C)"); ax2.grid(False)
ax1.set_title(f"Incidência semanal de dengue e temperatura mínima — estado de {UF_SIGLA}")
plt.show()

anual = pd.DataFrame({"casos": CASOS.sum(axis=1), "ano": ANO_EPI}).groupby("ano").casos.sum()
anual = pd.DataFrame({"casos": anual, "incidência / 100 mil": anual / pop_anual.loc[anual.index].sum(axis=1) * 1e5})
anual.round(0).astype(int)

### 6.2 Sazonalidade

Cada linha é uma temporada (SE 41 a SE 40). O pico costuma ocorrer entre as SE 10 e 18 (março a maio).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
for s in sorted(set(TEMPORADA)):
    l = np.where(TEMPORADA == s)[0]
    if len(l) >= 52:
        ax.plot(range(len(l)), inc_estado[l], lw=1.2, label=f"{s}/{str(s + 1)[2:]}")
pos = list(range(0, 52, 4))
ax.set_xticks(pos, [((40 + i) % 52) + 1 for i in pos])
ax.set(xlabel="semana epidemiológica", ylabel="casos / 100 mil hab.", title="Incidência estadual por temporada")
ax.legend(ncol=2, fontsize=8); plt.show()

### 6.3 Distribuição espacial

Incidência acumulada por município em cada ano (escala logarítmica).

In [ ]:
def desenhar_mapa(ax, valores, titulo, norma, cmap="YlOrRd"):
    patches, cores = [], []
    for g, v in zip(muns.geocodigo, valores):
        partes = getattr(geoms[g], "geoms", [geoms[g]])
        for p in partes:
            patches.append(MplPolygon(np.asarray(p.exterior.coords), closed=True))
            cores.append(v)
    pc = PatchCollection(patches, cmap=cmap, norm=norma, edgecolor="white", linewidth=0.1)
    pc.set_array(np.asarray(cores))
    ax.add_collection(pc)
    ax.autoscale_view(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(titulo)
    return pc

anos_mapa = [a for a in (2015, 2019, 2022, 2023, 2024, 2025) if a in set(ANO_EPI)]
acum = {a: CASOS[ANO_EPI == a].sum(axis=0) / pop_anual.loc[a].to_numpy() * 1e5 for a in anos_mapa}
norma = mcolors.LogNorm(vmin=1, vmax=max(np.nanmax(v) for v in acum.values()))
fig, axs = plt.subplots(2, 3, figsize=(16, 8))
for ax, a in zip(axs.ravel(), anos_mapa):
    pc = desenhar_mapa(ax, np.clip(acum[a], 1, None), str(a), norma)
fig.colorbar(pc, ax=axs, shrink=0.6, label="casos / 100 mil hab. no ano")
plt.show()

top = (pd.DataFrame(acum, index=muns.nome)[POP[-1] >= 100_000]
         .apply(lambda s: ", ".join(f"{n} ({v:,.0f})" for n, v in s.nlargest(3).items())))
print("Maiores incidências entre municípios com mais de 100 mil habitantes:\n" + top.to_string())

### 6.4 Defasagem entre clima e casos

O clima não age de imediato: o mosquito precisa se desenvolver, e o vírus precisa de tempo de incubação no vetor e na pessoa. A correlação de Spearman entre o clima da semana `t − lag` e a incidência da semana `t` (período de treino, todos os municípios) mostra quantas semanas de defasagem são mais informativas.

In [ ]:
lin_treino = np.where(semanas <= TREINO_ATE)[0]
correl = {}
for v in ["temp_min", "temp_med", "umid_med", "umid_max"]:
    X, Y = CLIMA[v][lin_treino], INC[lin_treino]
    correl[v] = [stats.spearmanr(X[: len(X) - L].ravel(), Y[L:].ravel(), nan_policy="omit")[0] for L in range(13)]
ax = pd.DataFrame(correl).plot(marker="o", figsize=(9, 4))
ax.set(xlabel="defasagem (semanas)", ylabel="ρ de Spearman", title="Correlação entre clima (t − lag) e incidência (t)")
plt.show()

## 7. Número reprodutivo efetivo (Rt)

O Rt é o número médio de pessoas infectadas por cada caso. **Rt > 1** indica que a transmissão está crescendo; **Rt < 1**, que está diminuindo.

Usamos o método de **Cori et al. (2013)**, implementado no pacote EpiEstim:

$$\Lambda_t = \sum_{s=1}^{6} w_s\, I_{t-s}, \qquad R_t \sim \text{Gama}\!\left(a_0 + \sum_{j=t-2}^{t} I_j,\ \ \text{taxa} = \frac{1}{b_0} + \sum_{j=t-2}^{t} \Lambda_j\right)$$

- $I_t$: casos na semana $t$;
- $w_s$: **intervalo de geração** da dengue (tempo entre a infecção de uma pessoa e a de quem ela infecta), discretizado em semanas: normal com média 3 e desvio 1 semana;
- janela de 3 semanas e distribuição *a priori* Gama($a_0 = 1$, $b_0 = 5$);
- $p_1 = P(R_t > 1)$: probabilidade de transmissão crescente. O InfoDengue considera **$p_1 > 0{,}95$** como evidência de transmissão sustentada.

In [ ]:
def deslocar(X, s):
    """Desloca uma matriz (semanas × municípios) s semanas para baixo (s > 0) ou para cima (s < 0)."""
    Y = np.full(X.shape, np.nan)
    if s > 0:
        Y[s:] = X[:-s]
    elif s < 0:
        Y[:s] = X[-s:]
    else:
        Y[:] = X
    return Y


def soma_movel(X, n):
    return pd.DataFrame(X).rolling(n, min_periods=n).sum().to_numpy()


W_GI = stats.norm.pdf(np.arange(1, 7), 3, 1)
W_GI /= W_GI.sum()


def estimar_rt(I, janela=3, a0=1.0, b0=5.0):
    lam = sum(W_GI[s - 1] * deslocar(I, s) for s in range(1, len(W_GI) + 1))
    forma, taxa = a0 + soma_movel(I, janela), 1 / b0 + soma_movel(lam, janela)
    return forma / taxa, stats.gamma.sf(1, forma, scale=1 / taxa)


RT, P1 = estimar_rt(CASOS)

n = 156
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].bar(semanas[-n:], CASOS[-n:, CAPITAL], width=5, color="#c0392b")
ax[0].set(title=f"{muns.nome[CAPITAL]} — casos por semana e Rt", ylabel="casos")
ax[1].plot(semanas[-n:], RT[-n:, CAPITAL], color="k", lw=1)
ax[1].axhline(1, color="grey", ls="--")
ax[1].fill_between(semanas[-n:], 0, 3, where=P1[-n:, CAPITAL] > 0.95, color="orange", alpha=0.3, label="P(Rt > 1) > 0,95")
ax[1].set(ylim=(0, 3), ylabel="Rt"); ax[1].legend()
plt.tight_layout(); plt.show()

frac = pd.Series((P1 > 0.95).mean(axis=1), index=semanas)
frac.plot(figsize=(13, 3), color="#e67e22", title="Fração de municípios com P(Rt > 1) > 0,95")
plt.show()

## 8. Limiares epidêmicos

Os limiares definem o que é incidência **normal**, **elevada** e **epidêmica** para cada município. O InfoDengue usa uma adaptação do **Moving Epidemic Method** (MEM, Vega et al., 2013). Implementamos uma versão do método:

1. Em cada temporada, o **período epidêmico** é o bloco de `r` semanas consecutivas com a maior soma de incidência. O tamanho `r` é escolhido pela curva MAP (porcentagem acumulada máxima): para quando acrescentar mais uma semana aumenta a porcentagem em menos de 2,8%.
2. As semanas antes do bloco são **pré-epidêmicas**, e as depois, **pós-epidêmicas**.
3. De cada temporada se tomam os `n ≈ 30 / nº de temporadas` maiores valores de cada fase.

| Limiar | Cálculo |
|---|---|
| `limiar_pre` | limite superior do IC 95% unilateral da média aritmética dos maiores valores pré-epidêmicos |
| `limiar_pos` | idem, com os valores pós-epidêmicos |
| `limiar_epidemico` | limite superior do IC 90% unilateral da média geométrica dos maiores valores epidêmicos |

Os limiares usam **apenas temporadas completas do período de treino**, para não vazar informação para a avaliação. Municípios com menos de 3 temporadas com pelo menos 20 casos recebem a mediana dos limiares dos demais.

In [ ]:
def ls_media(v, nivel):
    v = np.asarray(v, float)
    return np.nan if len(v) < 3 else v.mean() + stats.t.ppf(nivel, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))


def ls_geometrica(v, nivel):
    v = np.log(np.asarray(v, float)[np.asarray(v, float) > 0])
    return np.nan if len(v) < 3 else float(np.exp(v.mean() + stats.t.ppf(nivel, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))))


def limiares_mem(inc, casos, temporadas, delta=2.8, min_casos=20, min_temporadas=3):
    n_top = max(1, round(30 / len(temporadas)))
    fases = {f: [[] for _ in range(M)] for f in ("pre", "epi", "pos")}
    n_temp = np.zeros(M, int)
    for s in temporadas:
        linhas = np.where(TEMPORADA == s)[0]
        X, L = np.nan_to_num(inc[linhas]), len(linhas)
        total = X.sum(axis=0)
        acum = np.vstack([np.zeros(M), X.cumsum(axis=0)])
        max_soma, inicio = np.zeros((L, M)), np.zeros((L, M), int)
        for r in range(1, L + 1):                       # maior soma em r semanas seguidas
            somas = acum[r:] - acum[:-r]
            max_soma[r - 1], inicio[r - 1] = somas.max(axis=0), somas.argmax(axis=0)
        mapc = 100 * max_soma / np.where(total > 0, total, np.nan)
        incremento = np.diff(np.vstack([np.zeros(M), mapc]), axis=0)
        abaixo = np.nan_to_num(incremento, nan=0) < delta
        r_otimo = np.where(abaixo.any(axis=0), np.maximum(1, abaixo.argmax(axis=0)), L)
        for m in np.where(casos[linhas].sum(axis=0) >= min_casos)[0]:
            r, i0 = r_otimo[m], inicio[r_otimo[m] - 1, m]
            for f, vals in (("pre", X[:i0, m]), ("epi", X[i0:i0 + r, m]), ("pos", X[i0 + r:, m])):
                fases[f][m].extend(np.sort(vals)[-n_top:])
            n_temp[m] += 1
    lim = pd.DataFrame({"limiar_pre": [ls_media(v, 0.95) for v in fases["pre"]],
                        "limiar_pos": [ls_media(v, 0.95) for v in fases["pos"]],
                        "limiar_epidemico": [ls_geometrica(v, 0.90) for v in fases["epi"]],
                        "n_temporadas": n_temp}, index=muns.geocodigo)
    proprio = lim.n_temporadas >= min_temporadas
    for c in ["limiar_pre", "limiar_pos", "limiar_epidemico"]:
        lim.loc[~proprio | lim[c].isna(), c] = lim.loc[proprio, c].median()
    lim["limiar_epidemico"] = np.maximum(lim.limiar_epidemico, lim.limiar_pre)
    lim["limiar_proprio"] = proprio
    return lim


TEMPORADAS_TREINO = [int(s) for s in sorted(set(TEMPORADA))
                     if (TEMPORADA == s).sum() >= 52 and semanas[TEMPORADA == s].max() <= TREINO_ATE]
LIM = limiares_mem(INC, CASOS, TEMPORADAS_TREINO)
print(f"Temporadas de treino: {TEMPORADAS_TREINO}")
print(f"Municípios com limiares próprios: {LIM.limiar_proprio.sum()} de {M}")
LIM.join(muns.set_index("geocodigo").nome)[POP[-1] >= 100_000].sort_values("limiar_epidemico", ascending=False).head(10).round(2)

In [ ]:
# Diagrama de controle: temporadas da capital com os limiares
m = CAPITAL
fig, ax = plt.subplots(figsize=(12, 4.5))
for s in sorted(set(TEMPORADA)):
    l = np.where(TEMPORADA == s)[0]
    if len(l) >= 52:
        ax.plot(range(len(l)), INC[l, m], lw=2 if s > TEMPORADAS_TREINO[-1] else 0.8,
                alpha=1 if s > TEMPORADAS_TREINO[-1] else 0.5, label=f"{s}/{str(s + 1)[2:]}")
for c, cor in [("limiar_pre", "green"), ("limiar_pos", "orange"), ("limiar_epidemico", "red")]:
    ax.axhline(LIM.iloc[m][c], color=cor, ls="--", label=c)
pos = list(range(0, 52, 4))
ax.set_xticks(pos, [((40 + i) % 52) + 1 for i in pos])
ax.set_yscale("symlog", linthresh=1)
ax.set(xlabel="semana epidemiológica", ylabel="casos / 100 mil hab.",
       title=f"{muns.nome[m]} — incidência por temporada (linhas grossas: fora do treino) e limiares")
ax.legend(ncol=4, fontsize=7); plt.show()

acima = pd.Series(((INC > LIM.limiar_epidemico.values) & (CASOS > 10)).sum(axis=1), index=semanas)
acima.plot(figsize=(13, 3), color="#c0392b", title="Número de municípios acima do limiar epidêmico")
plt.show()

## 9. Formulação dos modelos

### 9.1 Problema

Em cada semana de referência `T` (o "hoje" de quem faz a previsão), queremos prever a semana `T + h`, com `h` = 1 e 4:

- **Regressão**: incidência de dengue na semana `T + h`, com uma previsão central e uma **faixa de incerteza**;
- **Classificação**: se a semana `T + h` estará **acima do limiar epidêmico** do município (incidência > `limiar_epidemico` e mais de 10 casos).

### 9.2 Variáveis preditoras

Todas usam apenas informação disponível na semana `T`:

| Grupo | Variáveis |
|---|---|
| Casos recentes | incidência das semanas `T` a `T−3` **com os casos já digitados até `T`**, a média das 4 e a tendência (últimas 2 semanas ÷ 2 anteriores) |
| Vizinhança | incidência recente média dos **municípios vizinhos** (que fazem divisa) |
| Rt | Rt e P(Rt > 1) calculados com os casos já digitados até `T` |
| Limiares | log dos limiares pré-epidêmico e epidêmico do município; razão entre a incidência recente e o limiar epidêmico |
| Sazonalidade | incidência da mesma semana no ano anterior; seno e cosseno da SE da semana-alvo |
| Clima defasado | `temp_min`, `temp_med`, `temp_max`, `umid_med`, `umid_max`: semanas `T−1` a `T−4`, média de `T−1` a `T−4`, média de `T−5` a `T−8` e média de `T−9` a `T−12` |
| Município | log da população |

- **Casos já digitados até `T`**: os casos das últimas semanas ainda estão chegando ao sistema. Usar a contagem final faria o modelo ver, no treino e no teste, dados que ninguém teria na hora de prever. A matriz `C[t, m, k]` reconstrói exatamente o que se conhecia em cada semana.
- **Vizinhos**: a dengue se espalha entre municípios próximos, e a situação dos vizinhos ajuda a antecipar a chegada de uma epidemia. Modelos que incluem a transmissão em áreas vizinhas tiveram melhor desempenho em estudos recentes para o Brasil (Chen & Moraga, 2025).
- **Defasagens longas**: a análise exploratória (seção 6.4) mostrou que a temperatura mínima tem a maior correlação com a incidência de 8 a 12 semanas depois.
- **O ano não é usado** como variável: modelos de árvore não extrapolam valores fora do intervalo de treino.

### 9.3 Prever o crescimento, não o nível

Modelos de árvore não preveem valores maiores do que os vistos no treino. Isso fez todos os modelos subestimarem a epidemia de 2024, muito maior que qualquer ano anterior. A solução é prever o **crescimento em relação ao nível atual**:

$$r = \frac{\text{incidência}_{T+h} + 1}{\text{incidência recente}_T + 1}, \qquad \widehat{\text{incidência}}_{T+h} = (\text{incidência recente}_T + 1)\cdot \hat r - 1$$

Uma razão como "a incidência vai triplicar em 4 semanas" já aparece no treino. Multiplicada por um nível recente alto, ela gera uma previsão acima de tudo o que foi visto antes.

### 9.4 Avaliação em janelas móveis

Em vez de um único corte, o modelo é **re-treinado a cada ano**, com todos os dados disponíveis até ali (janela expansível), e testado no ano seguinte:

| Ano de teste | Treino (semanas-alvo) |
|---|---|
| 2023 | até 2022 |
| 2024 | até 2023 |
| 2025 | até 2024 |

É assim que um sistema de previsão funciona na prática: a cada temporada, o modelo aprende com os anos mais recentes. Os limiares epidêmicos continuam calculados só com as temporadas até 2021, para que a definição de "semana epidêmica" seja a mesma em todos os anos de teste.

In [ ]:
from shapely.strtree import STRtree


def plano(A):
    """(semanas × municípios) -> vetor, na ordem das linhas do painel."""
    return np.asarray(A, float).ravel()


def casos_vistos_em_T(j):
    """Casos da semana T−j que já estavam digitados na semana T."""
    return deslocar(C[:, :, min(j, MAX_ATRASO)].astype(float), j)


def rt_aparente(janela=3, a0=1.0, b0=5.0, fim=1):
    """Rt com os casos conhecidos em T, janela terminando em T−fim."""
    V = {j: casos_vistos_em_T(j) for j in range(fim + janela + len(W_GI))}
    soma_i = sum(V[fim + i] for i in range(janela))
    soma_l = sum(W_GI[s - 1] * V[fim + i + s] for i in range(janela) for s in range(1, len(W_GI) + 1))
    forma, taxa = a0 + soma_i, 1 / b0 + soma_l
    return forma / taxa, stats.gamma.sf(1, forma, scale=1 / taxa)


# Vizinhança: municípios que fazem divisa (ilhas e casos isolados recebem os 3 centroides mais próximos)
lista_geoms = [geoms[g] for g in muns.geocodigo]
centroides = np.array([[p.centroid.x, p.centroid.y] for p in lista_geoms])
arvore = STRtree(lista_geoms)
W_VIZ = np.zeros((M, M), dtype=np.float32)
for i, g in enumerate(lista_geoms):
    viz = [j for j in arvore.query(g.buffer(0.002), predicate="intersects") if j != i]
    if not viz:
        d = ((centroides - centroides[i]) ** 2).sum(axis=1); d[i] = np.inf
        viz = list(np.argsort(d)[:3])
    W_VIZ[i, viz] = 1
W_VIZ /= W_VIZ.sum(axis=1, keepdims=True)
print(f"Vizinhos por município: média {(W_VIZ > 0).sum(axis=1).mean():.1f}")

RT_AP, P1_AP = rt_aparente()
LIM_PRE, LIM_EPI = LIM.limiar_pre.to_numpy(), LIM.limiar_epidemico.to_numpy()
VARS_MODELO_CLIMA = ["temp_min", "temp_med", "temp_max", "umid_med", "umid_max"]
INC_REC = {j: casos_vistos_em_T(j) / POP * 1e5 for j in range(4)}
INC_REC_MEDIA = np.mean([INC_REC[j] for j in range(4)], axis=0)
C_INC = 1.0                                        # constante da razão de crescimento (casos / 100 mil)


def montar_painel(h):
    col = {f"inc_recente_{j}": plano(INC_REC[j]) for j in range(4)}
    col["inc_recente_media4"] = plano(INC_REC_MEDIA)
    col["tendencia_recente"] = plano(np.log((INC_REC[0] + INC_REC[1] + 2 * C_INC) / (INC_REC[2] + INC_REC[3] + 2 * C_INC)))
    col["inc_vizinhos_media4"] = plano(INC_REC_MEDIA @ W_VIZ.T)
    col["rt"] = plano(np.clip(RT_AP, 0, 10))
    col["p_rt_maior_1"] = plano(P1_AP)
    col["log_limiar_pre"] = plano(np.log1p(np.broadcast_to(LIM_PRE, (T, M))))
    col["log_limiar_epi"] = plano(np.log1p(np.broadcast_to(LIM_EPI, (T, M))))
    col["razao_limiar_epi"] = col["inc_recente_media4"] / plano(np.broadcast_to(LIM_EPI, (T, M)))
    col["inc_ano_anterior"] = plano(deslocar(INC, 52 - h))                       # semana T+h−52 (já completa)
    col["inc_ano_anterior_3sem"] = plano(np.nanmean([deslocar(INC, 52 - h + d) for d in (-1, 0, 1)], axis=0))
    se_alvo = deslocar(SE[:, None].astype(float), -h)
    col["se_sin"] = plano(np.broadcast_to(np.sin(2 * np.pi * se_alvo / 52.18), (T, M)))
    col["se_cos"] = plano(np.broadcast_to(np.cos(2 * np.pi * se_alvo / 52.18), (T, M)))
    col["log_pop"] = plano(np.log(POP))
    for v in VARS_MODELO_CLIMA:
        defas = {L: deslocar(CLIMA[v], L) for L in range(1, 13)}
        for L in range(1, 5):
            col[f"{v}_lag{L}"] = plano(defas[L])
        col[f"{v}_media_1a4"] = plano(np.nanmean([defas[L] for L in range(1, 5)], axis=0))
        col[f"{v}_media_5a8"] = plano(np.nanmean([defas[L] for L in range(5, 9)], axis=0))
        col[f"{v}_media_9a12"] = plano(np.nanmean([defas[L] for L in range(9, 13)], axis=0))
    df = pd.DataFrame(col)
    df["m"] = np.tile(np.arange(M), T)
    df["semana_alvo"] = np.repeat(pd.Series(semanas).shift(-h).to_numpy(), M)
    df["pop_alvo"] = plano(deslocar(POP, -h))
    df["y"] = plano(deslocar(INC, -h))
    df["evento"] = plano((deslocar(INC, -h) > LIM_EPI) & (deslocar(CASOS, -h) > 10))
    return df


COLS_EXTRA = ("m", "semana_alvo", "pop_alvo", "y", "evento")


def preparar(df):
    feats = [c for c in df.columns if c not in COLS_EXTRA]
    base = df[df.y.notna() & df.inc_recente_3.notna() & df.inc_ano_anterior.notna() & df.p_rt_maior_1.notna()].copy()
    ref = base.semana_alvo <= TREINO_ATE
    for c in [c for c in feats if c.startswith(("temp", "umid"))]:     # clima ausente -> média até 2021
        base[c] = base[c].fillna(base.loc[ref, c].mean())
    return feats, base


ANOS_TESTE = [2023, 2024, 2025]


def janela(base, ano):
    """Treino: semanas-alvo até 31/12 do ano anterior. Teste: semanas-alvo do ano."""
    fim_treino = pd.Timestamp(f"{ano - 1}-12-31")
    return (base[base.semana_alvo <= fim_treino],
            base[(base.semana_alvo > fim_treino) & (base.semana_alvo <= pd.Timestamp(f"{ano}-12-31"))])


def variaveis_lineares(df):
    """Conjunto enxuto e não redundante para os modelos lineares, com log nas variáveis muito assimétricas."""
    return pd.DataFrame({
        "log_inc_recente": np.log1p(df.inc_recente_media4),
        "tendencia_recente": df.tendencia_recente,
        "log_inc_vizinhos": np.log1p(df.inc_vizinhos_media4),
        "log_inc_ano_anterior": np.log1p(df.inc_ano_anterior_3sem),
        "log_razao_limiar_epi": np.log1p(df.razao_limiar_epi),
        "p_rt_maior_1": df.p_rt_maior_1,
        "se_sin": df.se_sin, "se_cos": df.se_cos,
        "log_pop": df.log_pop,
        "temp_min_1a4": df.temp_min_media_1a4, "temp_min_9a12": df.temp_min_media_9a12,
        "umid_med_1a4": df.umid_med_media_1a4, "umid_med_9a12": df.umid_med_media_9a12,
    }, index=df.index)


PAINEIS = {}
for h in HORIZONTES:
    FEATS, PAINEIS[h] = preparar(montar_painel(h))
    print(f"h = {h}: {len(PAINEIS[h]):,} linhas | {len(FEATS)} variáveis | "
          f"teste por ano: {[len(janela(PAINEIS[h], a)[1]) for a in ANOS_TESTE]}")

## 10. Regressão: incidência `h` semanas à frente

### 10.1 Modelos

| Modelo | Descrição |
|---|---|
| **Persistência** (*baseline*) | repete a média das últimas 4 semanas com os casos já digitados |
| **Sazonal** (*baseline*) | repete a mesma semana do ano anterior |
| **Regressão linear** | modelo linear (mínimos quadrados) sobre o log da razão de crescimento, com proteção contra extrapolação (ver abaixo). Usa um conjunto enxuto de 13 variáveis não redundantes (incidências em log, tendência, vizinhos, Rt, sazonalidade, população, temperatura mínima e umidade médias em 1–4 e 9–12 semanas), padronizadas. É o modelo mais simples e **interpretável**: cada coeficiente diz quanto o crescimento esperado muda quando a variável aumenta um desvio-padrão (seção 10.5). Variáveis muito correlacionadas entre si foram deixadas de fora, porque tornariam os coeficientes instáveis |
| **GB Poisson (nível)** | Gradient Boosting com perda de Poisson prevendo diretamente a incidência |
| **GB Poisson (crescimento)** | Gradient Boosting com perda de Poisson prevendo a razão de crescimento *r* (seção 9.3) |
| **RF (crescimento)** | Random Forest prevendo a razão de crescimento *r* |
| **Ensemble** | média das três previsões acima. Combinar modelos foi o que funcionou melhor no desafio InfoDengue-Mosqlimate, principalmente em surtos extremos |
| **Quantis** | Gradient Boosting com perda quantílica (quantis 5%, 25%, 50%, 75% e 95% da razão de crescimento), que gera faixas de 50% e 90% |

### 10.2 Métricas

Calculadas em **número de casos** (incidência × população ÷ 100 mil):

- **MAE**: erro absoluto médio; **RMSE**: raiz do erro quadrático médio, que pesa mais os erros grandes;
- **R²**: fração da variação explicada (1 = perfeito; 0 = igual a prever a média; negativo = pior que a média);
- por **município-semana** e para o **total do estado** em cada semana;
- **MAE relativo**: MAE do modelo ÷ MAE da persistência (abaixo de 1 = melhor que o baseline);
- **WIS** (*Weighted Interval Score*): métrica usada no desafio InfoDengue-Mosqlimate para previsões com intervalo. Combina o erro da mediana com a largura das faixas e a penalidade quando o valor real cai fora delas. Para uma previsão sem intervalo, o WIS é igual ao erro absoluto, então pode ser comparado com o MAE dos outros modelos;
- **Cobertura**: fração das semanas em que o valor real caiu dentro da faixa de 50% e da de 90% (o ideal é ficar perto de 50% e 90%).

O treino completo (2 horizontes × 3 anos de teste) leva cerca de **30 a 45 minutos** no Colab. Os resultados ficam salvos no Drive (`dados/resultados_modelos`), e nas execuções seguintes esta etapa leva segundos. Para treinar de novo, por exemplo depois de mudar variáveis ou modelos, use `RETREINAR_MODELOS = True` na seção 3.

In [ ]:
QUANTIS = [0.05, 0.25, 0.5, 0.75, 0.95]
MODELOS_PONTO = ["GB Poisson (nível)", "GB Poisson (crescimento)", "RF (crescimento)"]


def novos_regressores():
    gb = dict(loss="poisson", max_iter=300, learning_rate=0.08, max_leaf_nodes=63, min_samples_leaf=50,
              l2_regularization=1.0, early_stopping=False, random_state=RANDOM_STATE)
    return {"GB Poisson (nível)": HistGradientBoostingRegressor(**gb),
            "GB Poisson (crescimento)": HistGradientBoostingRegressor(**gb),
            "RF (crescimento)": RandomForestRegressor(n_estimators=80, min_samples_leaf=30, max_features=0.5,
                                                      max_samples=0.2, n_jobs=-1, random_state=RANDOM_STATE)}


def para_razao(y, base):
    return np.clip((y + C_INC) / (base + C_INC), 0, 100)


def de_razao(r, base):
    return np.clip((base + C_INC) * r - C_INC, 0, None)


def wis(y, q):
    """WIS com mediana e intervalos de 50% e 90%; y (n,), q (n, 5) nos QUANTIS."""
    m = q[:, 2]
    total = 0.5 * np.abs(y - m)
    for alfa, (lo, hi) in ((0.1, (0, 4)), (0.5, (1, 3))):
        l, u = q[:, lo], q[:, hi]
        intervalo = (u - l) + 2 / alfa * np.clip(l - y, 0, None) + 2 / alfa * np.clip(y - u, 0, None)
        total += alfa / 2 * intervalo
    return total / 2.5


import joblib

DIR_RES = DIR_DADOS / "resultados_modelos"
DIR_RES.mkdir(parents=True, exist_ok=True)
RES, ULTIMOS_MODELOS = {}, {}
for h in HORIZONTES:
    arq_res, arq_mod = DIR_RES / f"regressao_h{h}.parquet", DIR_RES / f"modelos_h{h}.joblib"
    if arq_res.exists() and arq_mod.exists() and not RETREINAR_MODELOS:
        RES[h], ULTIMOS_MODELOS[h] = pd.read_parquet(arq_res), joblib.load(arq_mod)
        print(f"h = {h}: resultados carregados de {DIR_RES} (use RETREINAR_MODELOS = True para treinar de novo)")
        continue
    partes = []
    for ano in ANOS_TESTE:
        tr, te = janela(PAINEIS[h], ano)
        base_tr, base_te = tr.inc_recente_media4.to_numpy(), te.inc_recente_media4.to_numpy()
        r_tr = para_razao(tr.y.to_numpy(), base_tr)
        out = te[["semana_alvo", "m", "pop_alvo", "y", "evento", "razao_limiar_epi"]].copy()
        out["ano_teste"] = ano
        out["Persistência"] = base_te
        out["Sazonal (ano anterior)"] = te.inc_ano_anterior.to_numpy()
        modelos = novos_regressores()
        for nome, mod in modelos.items():
            if "nível" in nome:
                mod.fit(tr[FEATS], tr.y)
                out[nome] = np.clip(mod.predict(te[FEATS]), 0, None)
            else:
                mod.fit(tr[FEATS], r_tr)
                out[nome] = de_razao(mod.predict(te[FEATS]), base_te)
        out["Ensemble"] = out[MODELOS_PONTO].mean(axis=1)
        qs = []
        for q in QUANTIS:
            mq = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=150, learning_rate=0.1,
                                               max_leaf_nodes=31, min_samples_leaf=100, random_state=RANDOM_STATE)
            mq.fit(tr[FEATS], r_tr)
            qs.append(de_razao(mq.predict(te[FEATS]), base_te))
        qs = np.sort(np.column_stack(qs), axis=1)             # garante quantis em ordem
        for q, col in zip(QUANTIS, qs.T):
            out[f"q{int(q * 100):02d}"] = col
        partes.append(out)
        ULTIMOS_MODELOS[h] = (modelos, te)
        print(f"h = {h}, teste {ano}: treino {len(tr):,} linhas, teste {len(te):,} — pronto", flush=True)
    RES[h] = pd.concat(partes, ignore_index=True)
    RES[h].to_parquet(arq_res, index=False)
    joblib.dump(ULTIMOS_MODELOS[h], arq_mod, compress=3)

#### Regressão linear

A regressão linear é ajustada à parte, porque é rápida (não entra no cache acima). Modelos lineares extrapolam sem limite: se em um ano de teste uma variável passa muito do que existia no treino, como a incidência em 2024, a previsão pode explodir. Por isso há duas proteções usuais:
- as variáveis são limitadas ao intervalo observado no treino (percentis 0,1% e 99,9%);
- a previsão do log do crescimento também é limitada ao intervalo do treino.

Ao voltar da escala log, aplica-se a correção de viés de Duan (média das exponenciais dos resíduos, sem os 1% mais extremos de cada lado).

In [ ]:
def ajustar_linear(tr, te):
    Xtr, Xte = variaveis_lineares(tr), variaveis_lineares(te)
    lo, hi = Xtr.quantile(0.001), Xtr.quantile(0.999)
    Xtr, Xte = Xtr.clip(lo, hi, axis=1), Xte.clip(lo, hi, axis=1)
    alvo = np.log(para_razao(tr.y.to_numpy(), tr.inc_recente_media4.to_numpy()))
    modelo = make_pipeline(SimpleImputer(), StandardScaler(), LinearRegression()).fit(Xtr, alvo)
    residuos = alvo - modelo.predict(Xtr)
    duan = np.mean(np.exp(np.clip(residuos, *np.quantile(residuos, [0.01, 0.99]))))
    log_r = np.clip(modelo.predict(Xte), *np.quantile(alvo, [0.001, 0.999]))
    return modelo, de_razao(np.exp(log_r) * duan, te.inc_recente_media4.to_numpy())


for h in HORIZONTES:
    previsoes = []
    for ano in ANOS_TESTE:
        tr, te = janela(PAINEIS[h], ano)
        linear, p = ajustar_linear(tr, te)
        previsoes.append(p)
    RES[h]["Regressão linear"] = np.concatenate(previsoes)     # mesma ordem das linhas de RES[h]
    ULTIMOS_MODELOS[h][0]["Regressão linear"] = linear
    print(f"h = {h}: regressão linear ajustada nas {len(ANOS_TESTE)} janelas")

In [ ]:
NOMES_PONTO = ["Persistência", "Sazonal (ano anterior)", "Regressão linear"] + MODELOS_PONTO + ["Ensemble"]


def metricas_regressao(d):
    fator = d.pop_alvo.to_numpy() / 1e5
    y = d.y.to_numpy() * fator
    tot_y = pd.Series(y).groupby(d.semana_alvo.to_numpy()).sum()
    linhas = {}
    for nome in NOMES_PONTO:
        pc = np.nan_to_num(d[nome].to_numpy()) * fator
        tot_p = pd.Series(pc).groupby(d.semana_alvo.to_numpy()).sum()
        linhas[nome] = {"MAE": mean_absolute_error(y, pc), "RMSE": np.sqrt(mean_squared_error(y, pc)),
                        "R²": r2_score(y, pc), "MAE estado": mean_absolute_error(tot_y, tot_p),
                        "R² estado": r2_score(tot_y, tot_p)}
    tab = pd.DataFrame(linhas).T
    tab["MAE relativo"] = tab["MAE"] / tab.loc["Persistência", "MAE"]
    qcasos = d[[f"q{int(q * 100):02d}" for q in QUANTIS]].to_numpy() * fator[:, None]
    tab["WIS"] = np.nan
    tab.loc["Quantis (mediana + faixas)", ["MAE", "WIS"]] = [mean_absolute_error(y, qcasos[:, 2]), wis(y, qcasos).mean()]
    for nome in NOMES_PONTO:
        tab.loc[nome, "WIS"] = tab.loc[nome, "MAE"]           # sem intervalo, WIS = erro absoluto
    tab.loc["Quantis (mediana + faixas)", "MAE relativo"] = tab.loc["Quantis (mediana + faixas)", "MAE"] / tab.loc["Persistência", "MAE"]
    return tab


def cobertura(d):
    y = d.y.to_numpy()
    return {"faixa 50%": np.mean((y >= d.q25) & (y <= d.q75)), "faixa 90%": np.mean((y >= d.q05) & (y <= d.q95))}


for h in HORIZONTES:
    d = RES[h]
    print(f"\n=== h = {h} semana(s) — teste 2023 a 2025 (cada ano previsto por um modelo treinado até o ano anterior) ===")
    display(metricas_regressao(d).round(3))
    print("Cobertura das faixas:", {k: f"{v:.0%}" for k, v in cobertura(d).items()})
    por_ano = pd.DataFrame({ano: metricas_regressao(d[d.ano_teste == ano])["MAE relativo"] for ano in ANOS_TESTE})
    print(f"MAE relativo à persistência, por ano de teste (h = {h}):")
    display(por_ano.round(3))

### 10.3 Previsões no período de teste

À esquerda, o total do estado. À direita, a capital, com as faixas de 50% e 90% do modelo quantílico. As linhas verticais marcam o início de cada ano, quando o modelo é re-treinado.

In [ ]:
for h in HORIZONTES:
    d = RES[h]
    fig, axs = plt.subplots(1, 2, figsize=(15, 4.2))
    soma = lambda x, dd: pd.Series(np.nan_to_num(dd[x].to_numpy()) * dd.pop_alvo.to_numpy() / 1e5).groupby(dd.semana_alvo.to_numpy()).sum()
    ax = axs[0]
    ax.plot(soma("y", d), color="k", lw=2, label="observado")
    for nome, estilo in [("Ensemble", "-"), ("GB Poisson (nível)", "--"), ("Persistência", ":")]:
        ax.plot(soma(nome, d), estilo, label=nome)
    ax.set(title=f"Estado de {UF_SIGLA} — {h} semana(s) à frente", ylabel="casos por semana")
    cap = d[d.m == CAPITAL].sort_values("semana_alvo")
    f = cap.pop_alvo.to_numpy() / 1e5
    ax = axs[1]
    ax.fill_between(cap.semana_alvo, cap.q05 * f, cap.q95 * f, color="#2980b9", alpha=0.15, label="faixa 90%")
    ax.fill_between(cap.semana_alvo, cap.q25 * f, cap.q75 * f, color="#2980b9", alpha=0.3, label="faixa 50%")
    ax.plot(cap.semana_alvo, cap.y * f, color="k", lw=2, label="observado")
    ax.plot(cap.semana_alvo, cap.Ensemble * f, color="#c0392b", label="Ensemble")
    ax.set(title=f"{muns.nome[CAPITAL]} — {h} semana(s) à frente", ylabel="casos por semana")
    for a in axs:
        for ano in ANOS_TESTE[1:]:
            a.axvline(pd.Timestamp(f"{ano}-01-01"), color="grey", ls=":", lw=0.8)
        a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

### 10.4 Importância das variáveis

A importância por **permutação** mede quanto o erro aumenta quando os valores de uma variável são embaralhados num conjunto que o modelo não viu. Aqui ela é calculada para o modelo de crescimento treinado até 2024, no ano de teste de 2025.

In [ ]:
h = HORIZONTES[-1]
modelos, te = ULTIMOS_MODELOS[h]
mod = modelos["GB Poisson (crescimento)"]
amostra = te.sample(min(20_000, len(te)), random_state=RANDOM_STATE)
r_amostra = para_razao(amostra.y.to_numpy(), amostra.inc_recente_media4.to_numpy())
imp = permutation_importance(mod, amostra[FEATS], r_amostra, n_repeats=5, random_state=RANDOM_STATE,
                             scoring="neg_mean_poisson_deviance", n_jobs=-1)
ax = pd.Series(imp.importances_mean, index=FEATS).sort_values().tail(20).plot.barh(figsize=(8, 6))
ax.set(title=f"Importância por permutação — GB Poisson (crescimento), h = {h}, teste 2025",
       xlabel="aumento do desvio de Poisson")
plt.show()

### 10.5 Interpretação da regressão linear

**Como ler as métricas** (tabelas da seção 10.2):
- **MAE** é o erro típico em número de casos por município e semana. Um MAE de 20 significa que, em média, a previsão erra 20 casos para mais ou para menos.
- **RMSE** pesa mais os erros grandes. Quando é muito maior que o MAE, os erros se concentram em poucas semanas de pico.
- **R²** mede a fração da variação explicada. Abaixo de 0, o modelo é pior que prever a média.
- **MAE relativo** abaixo de 1 indica que o modelo é melhor que simplesmente repetir as últimas semanas.

**Como ler os coeficientes** (gráfico abaixo): o modelo prevê o **logaritmo do crescimento** da incidência entre `T` e `T + h`, com todas as variáveis padronizadas (média 0 e desvio-padrão 1). Assim, um coeficiente β significa que, quando a variável aumenta **um desvio-padrão**, o crescimento esperado é multiplicado por $e^{\beta}$. Por exemplo, β = 0,10 equivale a um crescimento cerca de 10,5% maior. Coeficientes positivos aceleram a previsão de casos; negativos a freiam.

<small>Os coeficientes descrevem **associações** dentro do modelo, não causas. Ainda há alguma correlação entre as variáveis (por exemplo, entre temperatura e sazonalidade), então cada coeficiente deve ser lido "mantendo as demais constantes".</small>

In [ ]:
h = HORIZONTES[-1]
modelos, te = ULTIMOS_MODELOS[h]
linear = modelos["Regressão linear"]
coefs = pd.Series(linear[-1].coef_, index=variaveis_lineares(te.head(1)).columns)
top = coefs.reindex(coefs.abs().sort_values().index)
ax = top.plot.barh(figsize=(8, 6), color=np.where(top > 0, "#c0392b", "#2980b9"))
ax.axvline(0, color="k", lw=0.8)
ax.set(title=f"Regressão linear — coeficientes padronizados (h = {h}, treino até 2024)",
       xlabel="efeito no log do crescimento por +1 desvio-padrão")
plt.show()

d = RES[h]
fator = d.pop_alvo.to_numpy() / 1e5
y_c, p_c = d.y.to_numpy() * fator, d["Regressão linear"].to_numpy() * fator
print(f"Regressão linear, h = {h}, teste 2023–2025: MAE = {mean_absolute_error(y_c, p_c):.1f} casos | "
      f"RMSE = {np.sqrt(mean_squared_error(y_c, p_c)):.1f} casos | R² = {r2_score(y_c, p_c):.3f}")
print("Maiores efeitos (crescimento multiplicado por, a cada +1 desvio-padrão):")
for nome, b in coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(5).items():
    print(f"  {nome:28s} β = {b:+.3f}  →  ×{np.exp(b):.2f}")

## 11. Classificação: o município vai passar do limiar epidêmico?

Para a vigilância, antecipar a entrada de um município em **situação epidêmica** pode ser mais útil do que acertar o número exato de casos. O evento é *incidência > limiar epidêmico e mais de 10 casos* na semana `T + h`.

Como o evento é raro, as métricas são:
- **precisão**: das semanas em que o modelo previu epidemia, quantas de fato foram;
- **sensibilidade** (*recall*): das semanas epidêmicas, quantas o modelo antecipou;
- **F1**: média harmônica das duas;
- **AP** (*average precision*): resumo da curva precisão × sensibilidade, independente do ponto de corte.

Comparações (todas nas mesmas janelas móveis):
- **Persistência**: prevê epidemia se a incidência recente já está acima do limiar;
- **Regressão logística**: modelo linear para a probabilidade do evento, com as mesmas 13 variáveis da regressão linear, padronizadas, e pesos que equilibram as classes (o evento é raro). É o modelo de classificação mais simples e interpretável. O ponto de corte é o de maior F1 no próprio treino de cada janela;
- **Regressão (Ensemble)**: prevê epidemia se a incidência prevista pelo ensemble passa do limiar;
- **Gradient Boosting (classif.)**: treinado diretamente para o evento, com pesos que equilibram as classes. Ponto de corte 0,5.

In [ ]:
def metricas_classif(y, escore, corte):
    prev = np.nan_to_num(escore) > corte
    return {"precisão": precision_score(y, prev, zero_division=0), "sensibilidade": recall_score(y, prev, zero_division=0),
            "F1": f1_score(y, prev, zero_division=0), "AP": average_precision_score(y, np.nan_to_num(escore))}


CLF = {}
def treinar_classificadores(h):
    """Gradient Boosting e regressão logística em cada janela; devolve os escores do teste e a última logística."""
    escores, escores_log = [], []
    for ano in ANOS_TESTE:
        tr, te = janela(PAINEIS[h], ano)
        clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=50,
                                             class_weight="balanced", early_stopping=False, random_state=RANDOM_STATE)
        clf.fit(tr[FEATS], tr.evento)
        escores.append(clf.predict_proba(te[FEATS])[:, 1])
        logit = make_pipeline(SimpleImputer(), StandardScaler(),
                              LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0))
        logit.fit(variaveis_lineares(tr), tr.evento)
        # ponto de corte: maior F1 no próprio treino (modelo linear simples, pouco sujeito a sobreajuste)
        p_tr, r_tr_, c_tr = precision_recall_curve(tr.evento, logit.predict_proba(variaveis_lineares(tr))[:, 1])
        f1_tr = 2 * p_tr * r_tr_ / np.where(p_tr + r_tr_ > 0, p_tr + r_tr_, 1)
        corte = c_tr[np.argmax(f1_tr[:-1])]
        escores_log.append(logit.predict_proba(variaveis_lineares(te))[:, 1] / corte * 0.5)   # corte do ano vira 0,5
    return np.concatenate(escores), np.concatenate(escores_log), logit


LOGITS = {}
for h in HORIZONTES:
    arq_cl, arq_lg = DIR_RES / f"classificacao_h{h}.parquet", DIR_RES / f"logistica_h{h}.joblib"
    if arq_cl.exists() and arq_lg.exists() and not RETREINAR_MODELOS:
        salvo = pd.read_parquet(arq_cl)
        esc_gb, esc_log, LOGITS[h] = salvo.gb.to_numpy(), salvo.logistica.to_numpy(), joblib.load(arq_lg)
    else:
        esc_gb, esc_log, LOGITS[h] = treinar_classificadores(h)
        pd.DataFrame({"gb": esc_gb, "logistica": esc_log}).to_parquet(arq_cl, index=False)
        joblib.dump(LOGITS[h], arq_lg, compress=3)
    d = RES[h]
    sc = {"Persistência": (d.razao_limiar_epi.to_numpy(), 1.0),
          "Regressão (Ensemble)": (d.Ensemble.to_numpy() / LIM_EPI[d.m.to_numpy()], 1.0),
          "Regressão logística": (esc_log, 0.5),
          "Gradient Boosting (classif.)": (esc_gb, 0.5)}
    y = d.evento.to_numpy().astype(bool)
    tab = pd.DataFrame({nome: metricas_classif(y, s, c) for nome, (s, c) in sc.items()}).T
    f1_ano = pd.DataFrame({ano: {nome: metricas_classif(y[d.ano_teste == ano], s[d.ano_teste == ano], c)["F1"]
                                 for nome, (s, c) in sc.items()} for ano in ANOS_TESTE})
    CLF[h] = {"escores": {k: v[0] for k, v in sc.items()}, "cortes": {k: v[1] for k, v in sc.items()}, "y": y}
    print(f"\n=== h = {h} semana(s) — teste 2023 a 2025 ({int(y.sum()):,} semanas-município epidêmicas) ===")
    display(tab.round(3))
    print("F1 por ano de teste:")
    display(f1_ano.round(3))

In [ ]:
fig, axs = plt.subplots(1, len(HORIZONTES), figsize=(6 * len(HORIZONTES), 4.5), squeeze=False)
for ax, h in zip(axs[0], HORIZONTES):
    y = CLF[h]["y"]
    for nome, s in CLF[h]["escores"].items():
        p, r, _ = precision_recall_curve(y, np.nan_to_num(s))
        ax.plot(r, p, label=f"{nome} (AP = {average_precision_score(y, np.nan_to_num(s)):.2f})")
    ax.axhline(y.mean(), color="grey", ls=":", label="aleatório")
    ax.set(xlabel="sensibilidade", ylabel="precisão", title=f"Curva precisão × sensibilidade — h = {h}")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

### 11.1 Matrizes de confusão

Cada matriz mostra, no período de teste (2023 a 2025), quantas semanas-município foram:
- **verdadeiros negativos** (canto superior esquerdo): sem epidemia, sem alerta;
- **falsos positivos** (superior direito): alerta emitido, mas não houve epidemia (**alarme falso**);
- **falsos negativos** (inferior esquerdo): houve epidemia, mas não houve alerta (**epidemia não antecipada**);
- **verdadeiros positivos** (inferior direito): epidemia antecipada corretamente.

A **precisão** é VP ÷ (VP + FP), o **recall** (sensibilidade) é VP ÷ (VP + FN) e o **F1** é a média harmônica dos dois.

In [ ]:
modelos_matriz = ["Regressão logística", "Gradient Boosting (classif.)"]
fig, axs = plt.subplots(len(HORIZONTES), len(modelos_matriz), figsize=(5.5 * len(modelos_matriz), 4.6 * len(HORIZONTES)), squeeze=False)
for i, h in enumerate(HORIZONTES):
    y = CLF[h]["y"]
    for j, nome in enumerate(modelos_matriz):
        prev = np.nan_to_num(CLF[h]["escores"][nome]) > CLF[h]["cortes"][nome]
        cm = confusion_matrix(y, prev, labels=[False, True])
        ConfusionMatrixDisplay(cm, display_labels=["sem epidemia", "epidemia"]).plot(ax=axs[i, j], cmap="Blues", colorbar=False, values_format=",d")
        vn, fp, fn, vp = cm.ravel()
        axs[i, j].set_title(f"{nome} — h = {h}\nprecisão {vp / max(vp + fp, 1):.2f} | recall {vp / max(vp + fn, 1):.2f} | "
                            f"F1 {2 * vp / max(2 * vp + fp + fn, 1):.2f}", fontsize=10)
        axs[i, j].set(xlabel="previsto", ylabel="observado")
plt.tight_layout(); plt.show()

### 11.2 Interpretação da regressão logística

Os coeficientes da regressão logística (variáveis padronizadas) viram **razões de chance** (*odds ratios*): $e^{\beta}$ é o fator pelo qual a chance de a semana ser epidêmica é multiplicada quando a variável aumenta um desvio-padrão. Acima de 1 aumenta o risco; abaixo de 1 reduz.

In [ ]:
h = HORIZONTES[-1]
coefs_log = pd.Series(LOGITS[h][-1].coef_[0], index=variaveis_lineares(PAINEIS[h].head(1)).columns)
top = coefs_log.reindex(coefs_log.abs().sort_values().index)
ax = np.exp(top).plot.barh(figsize=(8, 6), color=np.where(top > 0, "#c0392b", "#2980b9"))
ax.axvline(1, color="k", lw=0.8)
ax.set_xscale("log")
ax.set(title=f"Regressão logística — razões de chance por +1 desvio-padrão (h = {h}, treino até 2024)",
       xlabel="razão de chance (escala log)")
plt.show()

## 12. Cenário sazonal com climatologia

Os modelos acima dependem dos casos recentes, então só preveem poucas semanas à frente. Para um horizonte longo (uma temporada inteira), o modelo abaixo usa apenas sazonalidade, perfil histórico do município e clima:

- seno/cosseno da SE e log da população;
- incidência média do município e incidência média do município naquela SE, calculadas **só com o treino**;
- clima defasado de 1 a 4 semanas.

No treino o modelo vê o **clima observado**. Para projetar o futuro, quando o clima ainda não é conhecido, usa-se a **climatologia**: a média histórica de cada variável, por município e semana epidemiológica, calculada **só com os anos de treino**. Também se mostra o cenário com o clima que de fato ocorreu, para medir quanto o conhecimento do clima melhora a projeção.

In [ ]:
lin_tr = semanas <= TREINO_ATE


def media_por_se(X):
    """Valor médio de cada município em cada SE no treino, repetido para todas as semanas."""
    df = pd.DataFrame(X[lin_tr]); df["se"] = SE[lin_tr]
    return df.groupby("se").mean().reindex(range(1, 54)).ffill().bfill().to_numpy()[SE - 1]


CLIMATOLOGIA = {v: media_por_se(CLIMA[v]) for v in VARS_MODELO_CLIMA}
PERFIL_SE = media_por_se(INC)
MEDIA_MUN = np.broadcast_to(np.nanmean(INC[lin_tr], axis=0), (T, M))


def painel_sazonal(clima_usado):
    col = {"se_sin": plano(np.broadcast_to(np.sin(2 * np.pi * SE / 52.18)[:, None], (T, M))),
           "se_cos": plano(np.broadcast_to(np.cos(2 * np.pi * SE / 52.18)[:, None], (T, M))),
           "log_pop": plano(np.log(POP)), "inc_media_mun": plano(MEDIA_MUN), "inc_media_mun_se": plano(PERFIL_SE)}
    for v in VARS_MODELO_CLIMA:
        for L in range(1, 5):
            col[f"{v}_lag{L}"] = plano(deslocar(clima_usado[v], L))
    return pd.DataFrame(col)


X_obs, X_clim = painel_sazonal(CLIMA), painel_sazonal(CLIMATOLOGIA)
F_SAZ = list(X_obs.columns)
semana_lin = np.repeat(semanas.to_numpy(), M)
y_lin, pop_lin = plano(INC), plano(POP)
m_tr = (semana_lin <= TREINO_ATE.to_datetime64()) & X_obs.notna().all(axis=1).to_numpy() & np.isfinite(y_lin)
m_fut = semana_lin > TREINO_ATE.to_datetime64()

modelo_saz = HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                           min_samples_leaf=100, early_stopping=False, random_state=RANDOM_STATE)
modelo_saz.fit(X_obs[m_tr], y_lin[m_tr])

proj = pd.DataFrame({"semana": semana_lin[m_fut],
                     "observado": y_lin[m_fut],
                     "climatologia": modelo_saz.predict(X_clim[m_fut]),
                     "clima observado": modelo_saz.predict(X_obs[m_fut].fillna(X_clim[m_fut]))})
for c in ["observado", "climatologia", "clima observado"]:
    proj[c] *= pop_lin[m_fut] / 1e5
tot = proj.groupby("semana").sum()

anos_proj = sorted(set(tot.index.year))
fig, axs = plt.subplots(1, len(anos_proj), figsize=(4.5 * len(anos_proj), 3.8), sharey=True)
for ax, ano in zip(np.atleast_1d(axs), anos_proj):
    d = tot[tot.index.year == ano]
    ax.plot(d.index, d["observado"], "k", lw=2, label="observado")
    ax.plot(d.index, d["climatologia"], color="#2980b9", label="projeção com climatologia")
    ax.plot(d.index, d["clima observado"], color="#e67e22", ls="--", label="cenário com clima observado")
    ax.set_title(str(ano)); ax.tick_params(axis="x", rotation=45)
np.atleast_1d(axs)[0].set_ylabel("casos por semana"); np.atleast_1d(axs)[0].legend(fontsize=8)
plt.suptitle(f"Projeção sazonal fora da amostra — total de {UF_SIGLA} (modelo ajustado até {TREINO_ATE.year})")
plt.tight_layout(); plt.show()

pd.DataFrame({c: {"MAE estado": mean_absolute_error(tot["observado"], tot[c]),
                  "R² estado": r2_score(tot["observado"], tot[c])} for c in ["climatologia", "clima observado"]}).T.round(3)

O modelo sazonal capta **quando** a temporada acontece, mas não **o tamanho** de cada epidemia. A intensidade de um ano depende de fatores ausentes desta base, como o sorotipo em circulação, a imunidade da população e as ações de controle. Por isso a projeção com climatologia deve ser lida como o **ano típico esperado**, e as previsões de curto prazo (seções 10 e 11) dependem dos casos recentes. (Esta seção usa um único corte, treino até 2021, porque o objetivo é medir o sinal climático isolado.)

## 13. Recomendações para a vigilância

Esta seção traduz as previsões em **ações**, com os números calculados no período de teste (2023 a 2025) para o horizonte de 4 semanas, que é o que dá tempo de agir.

In [ ]:
h = HORIZONTES[-1]
y = CLF[h]["y"]
d = RES[h]
resumo = {}
for nome in ["Regressão logística", "Gradient Boosting (classif.)", "Persistência"]:
    prev = np.nan_to_num(CLF[h]["escores"][nome]) > CLF[h]["cortes"][nome]
    vn, fp, fn, vp = confusion_matrix(y, prev, labels=[False, True]).ravel()
    resumo[nome] = {"semanas epidêmicas antecipadas (%)": 100 * vp / max(vp + fn, 1),
                    "alertas confirmados (%)": 100 * vp / max(vp + fp, 1),
                    "alarmes falsos por alerta certo": fp / max(vp, 1),
                    "epidemias não antecipadas": fn}
display(pd.DataFrame(resumo).T.round(2))

fator = d.pop_alvo.to_numpy() / 1e5
y_c = d.y.to_numpy() * fator
dentro90 = np.mean((d.y >= d.q05) & (d.y <= d.q95))
acima90 = np.mean(d.y > d.q95)
print(f"Faixa de 90% (h = {h}): contém o valor real em {dentro90:.0%} das semanas-município; "
      f"o real fica ACIMA da faixa em {acima90:.0%}.")
pico = (d.assign(observado=d.y * d.pop_alvo / 1e5, ensemble=d.Ensemble * d.pop_alvo / 1e5)
          .groupby("semana_alvo")[["observado", "ensemble"]].sum())
semana_pico = pico.observado.idxmax()
print(f"Pico do teste: semana de {semana_pico.date()} — observado {pico.observado.max():,.0f} casos, "
      f"ensemble previu {pico.loc[semana_pico, 'ensemble']:,.0f} com {h} semanas de antecedência "
      f"({pico.loc[semana_pico, 'ensemble'] / pico.observado.max():.0%} do real).")

### Ações recomendadas

1. **Alerta antecipado por município (classificação, 4 semanas).** Quando o modelo indicar probabilidade de semana epidêmica acima do ponto de corte, a secretaria municipal deve intensificar as ações **antes** do pico: mutirões de eliminação de criadouros, visitas casa a casa, levantamento de infestação (LIRAa) e bloqueio de transmissão em torno dos casos. Quatro semanas é aproximadamente o tempo de um ciclo do mosquito somado à incubação do vírus, então agir nessa janela reduz a transmissão que viria depois.
2. **Preparar a rede de saúde usando o limite superior da faixa, não a previsão central.** Leitos, centros de hidratação, equipes e insumos (soro, testes) devem ser dimensionados pelo **limite de 90%** da previsão, porque as epidemias recentes, sobretudo a de 2024, ficaram acima do esperado. Planejar pela média deixa a rede despreparada nas semanas de pico.
3. **Olhar a vizinhança.** A incidência dos municípios vizinhos está entre as variáveis mais importantes do modelo. Municípios com vizinhos em alta devem entrar em alerta mesmo antes de os próprios casos subirem, e as ações devem ser coordenadas por região de saúde, não só por município.
4. **Usar o clima como sinal de pré-temporada.** A temperatura mínima tem a maior correlação com a incidência de 8 a 12 semanas depois (seção 6.4). O aumento da temperatura mínima entre setembro e novembro deve disparar campanhas de prevenção e comunicação **antes** do início da temporada (SE 41).
5. **Priorizar recursos por risco relativo.** Ordenar os municípios pela razão entre a incidência prevista e o próprio limiar epidêmico ajuda a distribuir equipes e insumos estaduais para onde o risco relativo é maior, e não só para os municípios grandes.
6. **Modelo como apoio, não substituto.** As previsões devem ser revisadas semanalmente pela vigilância epidemiológica junto com dados de campo (infestação, sorotipo em circulação, ocupação de leitos), que o modelo não enxerga.

### Impactos e riscos

- **Epidemias não antecipadas (falsos negativos)** custam mais: resposta atrasada, rede sobrecarregada e mais casos graves e óbitos. Por isso o ponto de corte do classificador privilegia a sensibilidade, aceitando algum alarme falso.
- **Alarmes falsos (falsos positivos)** consomem equipes e insumos e, se forem frequentes, geram "fadiga de alerta": os gestores passam a ignorar os avisos. A tabela acima mostra quantos alarmes falsos há para cada alerta correto. Se for alto, vale subir o ponto de corte ou exigir alerta em 2 semanas seguidas, como faz o InfoDengue.
- **Anos fora do padrão** (como 2024, com sorotipo novo) tendem a ser **subestimados**. O modelo indica a direção certa, mas o tamanho real pode ser bem maior, e a gestão deve considerar esse risco.
- **Municípios pequenos** têm séries com poucos casos e limiares menos precisos. Os alertas para eles devem ser lidos com mais cautela, e a regionalização (item 3) ajuda.
- **Equidade:** a notificação depende do acesso ao serviço de saúde. Áreas com menos acesso podem ter casos subnotificados e, portanto, risco subestimado pelo modelo.

## 14. Conclusões

### Dados e indicadores

- A base reúne **6,8 milhões de notificações** de residentes nos 645 municípios de SP entre 2014 e 2025, com clima ERA5-Land completo para todas as semanas e municípios.
- **2024 foi um ano fora da curva**: 4.746 notificações por 100 mil habitantes, quase três vezes as epidemias anteriores mais fortes (1.671 em 2015 e 1.562 em 2019). No pico, cerca de 350 municípios estavam ao mesmo tempo acima do próprio limiar epidêmico, contra cerca de 250 em 2015 e 2019.
- Os mapas mostram que o **noroeste paulista** (São José do Rio Preto, Votuporanga, Birigui, Presidente Prudente) concentra a transmissão na maioria dos anos e tem os maiores limiares epidêmicos. Em 2024 a epidemia se espalhou para regiões normalmente menos atingidas, como o Vale do Paraíba: São José dos Campos, Jacareí e Pindamonhangaba tiveram as maiores incidências entre os municípios com mais de 100 mil habitantes.
- Os limiares foram estimados com dados próprios para 561 dos 645 municípios; os demais, com poucos casos no treino, receberam a mediana.
- O **Rt** se comportou como esperado: os períodos com P(Rt > 1) > 0,95 começam semanas antes dos picos de incidência. Nas temporadas epidêmicas (2014/15, 2018/19, 2023/24 e 2024/25), a fração de municípios com transmissão sustentada chegou a cerca de 70%.

### Clima

- A **temperatura mínima** é a variável climática mais associada à incidência. A correlação de Spearman sobe de 0,13 (mesma semana) para cerca de 0,32 com 10 a 12 semanas de defasagem. A umidade tem associação mais fraca (ρ ≈ 0,14), com máximo em torno de 6 semanas.
- Porém o clima explica **quando** a temporada acontece, não **o tamanho** da epidemia. A projeção sazonal teve R² estadual de apenas 0,05 com climatologia e 0,08 com o clima observado. Saber o clima real melhorou pouco a projeção, e em 2024 ela ficou muito abaixo do observado. A intensidade de cada ano depende de fatores fora desta base, como o sorotipo em circulação (em 2024, a reintrodução do DENV-3) e a imunidade da população.

### Modelos de previsão (seções 10 e 11)

**Regressão.** Com as janelas móveis (cada ano previsto por um modelo treinado até o ano anterior):

| Modelo | MAE relativo (1 sem.) | MAE relativo (4 sem.) | R² estado (4 sem.) |
|---|---|---|---|
| Persistência | 1,00 | 1,00 | 0,34 |
| Sazonal (ano anterior) | 2,34 | 1,76 | −0,44 |
| GB Poisson (nível) | 0,92 | 0,80 | 0,64 |
| GB Poisson (crescimento) | 0,81 | 0,79 | 0,57 |
| **RF (crescimento)** | **0,74** | **0,73** | **0,71** |
| Ensemble | 0,79 | 0,75 | 0,65 |

- O melhor modelo foi a **Random Forest prevendo o crescimento**, com erro 26% a 27% menor que a persistência nos dois horizontes. O ensemble ficou logo atrás e foi o mais **estável entre os anos** (MAE relativo de 0,73 a 0,76 em 2023, 2024 e 2025, com 4 semanas).
- **Prever o crescimento funcionou para o problema de 2024.** Com 4 semanas de antecedência, o ensemble previu cerca de 71 mil casos na semana de pico, contra 156 mil observados (45% do real). Antes, com um corte único e modelos de nível, a previsão ficava em torno de 30 mil. Ainda é uma subestimativa, mas o modelo passou a acompanhar a escala da epidemia.
- **Faixas de incerteza:** a faixa de 50% conteve o valor real em 56% das semanas e a de 90% em 87%, perto do ideal. O WIS do modelo quantílico (15,5 casos, com 4 semanas) foi menor que o erro de todos os modelos pontuais. Nas semanas em que o real fica fora da faixa de 90%, ele fica quase sempre **acima** (10% das semanas), outra vez pela subestimação dos picos.
- **Variáveis mais importantes** (permutação, teste 2025): razão entre a incidência recente e o limiar epidêmico, sazonalidade, **incidência dos vizinhos** e **temperatura mínima das últimas 4 semanas**, nessa ordem. A vizinhança, incluída nesta versão, ficou entre as quatro mais importantes.
- **Regressão linear:** é o modelo mais simples e interpretável, e serve de referência para os modelos de árvore. Os coeficientes (seção 10.5) têm interpretação epidemiológica clara: onde a incidência já está alta, o crescimento previsto é **menor** (a epidemia desacelera quando já atingiu muita gente), e **vizinhos em alta aceleram** o crescimento. Os números calculados nesta execução estão na célula a seguir.

In [ ]:
from IPython.display import Markdown

texto = ["**Regressão linear — resultados no teste 2023–2025:**", ""]
for h in HORIZONTES:
    tab = metricas_regressao(RES[h])
    t = tab.loc["Regressão linear"]
    comp = "menor" if t["MAE relativo"] < 1 else "maior"
    texto.append(f"- **{h} semana(s)**: MAE = {t['MAE']:.1f} casos, RMSE = {t['RMSE']:.0f} casos, R² = {t['R²']:.3f} "
                 f"(R² estadual {t['R² estado']:.2f}); erro {abs(1 - t['MAE relativo']):.0%} {comp} que a persistência. "
                 f"Melhor R² nesse horizonte: {tab['R²'].idxmax()} ({tab['R²'].max():.3f}).")
principais = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(4)
texto.append("- Maiores coeficientes (4 semanas, por +1 desvio-padrão): " +
             "; ".join(f"`{n}` β = {b:+.2f} (crescimento ×{np.exp(b):.2f})" for n, b in principais.items()) + ".")
display(Markdown("\n".join(texto)))


**Classificação** (semana epidêmica com 4 semanas de antecedência, teste 2023–2025, 12.275 semanas-município epidêmicas):

| Modelo | Precisão | Recall | F1 | AP |
|---|---|---|---|---|
| Persistência | 0,60 | 0,44 | 0,51 | 0,56 |
| Regressão logística | **0,73** | 0,60 | 0,66 | 0,70 |
| Gradient Boosting (classif.) | 0,54 | **0,86** | 0,66 | **0,78** |
| Regressão (Ensemble) | 0,72 | 0,66 | **0,69** | 0,77 |

- Todos os modelos superam a persistência com folga (F1 de 0,66–0,69 contra 0,51).
- Os modelos têm **perfis diferentes**, úteis para usos diferentes: a **regressão logística** é a mais precisa (73% dos alertas se confirmam, 0,37 alarme falso por alerta certo), e o **Gradient Boosting** é o que mais antecipa (86% das semanas epidêmicas, com 0,87 alarme falso por alerta certo).
- Na regressão logística, as maiores razões de chance são da sazonalidade, da temperatura mínima das últimas 4 semanas, da incidência recente, da população e dos vizinhos (cada uma aumenta a chance de epidemia em cerca de 1,7 a 2 vezes por desvio-padrão).
- Com 1 semana de antecedência, o F1 sobe para 0,74–0,78 em todos os modelos.

### Limitações

- **Atraso de notificação**: as variáveis usam só os casos já digitados na data da previsão, o que evita vazamento de dados, mas não há correção estatística (*nowcasting*) dos casos que ainda vão chegar.
- **População**: as estimativas do IBGE até 2021 são baseadas no Censo 2010 e somam 46,6 milhões em 2021, contra 44,4 milhões no Censo 2022. Por isso a incidência dos anos anteriores a 2022 está levemente subestimada.
- **Rt** com intervalo de geração fixo (o InfoDengue usa um intervalo que depende da temperatura) e **limiares** por uma versão simplificada do MEM, mantidos fixos (temporadas até 2021) em todas as janelas.
- **Casos notificados** incluem casos depois descartados.
- **Faixas de incerteza por município**: as faixas do total do estado não são calculadas, porque somar quantis de municípios não dá o quantil da soma.
- Não há informação sobre o **sorotipo** em circulação, que parece ser o principal fator da intensidade de 2024.

### Próximos passos

- Incluir a precipitação (disponível no mesmo conjunto ERA5-Land do Earth Engine).
- Testar modelos de redes neurais (LSTM) com embedding de município, comparando nas mesmas janelas móveis.
- Incluir informação sobre o sorotipo circulante e a cobertura vacinal.
- Corrigir o atraso de notificação com um modelo de *nowcasting*, como faz o InfoDengue.

## 15. Reprodutibilidade

In [ ]:
import platform, sklearn, scipy
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scikit-learn {sklearn.__version__} | scipy {scipy.__version__}")
print(f"Período: {semanas[0].date()} a {semanas[-1].date()} | municípios: {M} | casos: {CASOS.sum():,.0f}")
print(f"Executado em: {pd.Timestamp.now():%d/%m/%Y %H:%M}")

## 16. Exportar dados para o painel interativo

O repositório tem um **painel em Streamlit** (pasta `painel/`) para explorar as previsões por município, no mapa e no total do estado. O painel não treina modelos: ele lê os arquivos gerados nesta célula, que ficam em `MyDrive/dengue_sp/painel_arquivos` (menos de 25 MB) e são baixados num `.zip`. Para atualizar o painel, descompacte o `.zip` na pasta `painel/arquivos/` do repositório.

| Arquivo | Conteúdo |
|---|---|
| `municipios.parquet` | nome, população, limiares e centroide de cada município |
| `malha.geojson` | contornos simplificados dos municípios, para o mapa |
| `serie.parquet` | casos, incidência, Rt, temperatura mínima e umidade por semana e município |
| `previsoes_teste.parquet` | previsões de 2023 a 2025 (horizontes de 1 e 4 semanas), faixas de 50% e 90% e probabilidade de semana epidêmica |
| `previsao_futura.parquet` | previsão para 1 e 4 semanas depois da última semana da base, com os modelos do último ano de teste |
| `metricas.json` | métricas de regressão e classificação e informações gerais |

A **probabilidade de semana epidêmica** usa a distribuição dos erros relativos do ensemble no teste: para cada previsão, conta em que fração dos erros observados o valor real passaria do limiar epidêmico do município. As faixas da previsão futura usam os mesmos erros.

In [ ]:
import shapely
from shapely.geometry import mapping

DIR_PAINEL = DIR_DADOS.parent / "painel_arquivos"
DIR_PAINEL.mkdir(parents=True, exist_ok=True)
LIM_EPI_M = LIM.limiar_epidemico.to_numpy()
f32 = lambda x: np.asarray(x, dtype=np.float32)

# 1. Municípios (limiares, população, centroide) e malha simplificada
centro = np.array([[geoms[g].centroid.y, geoms[g].centroid.x] for g in muns.geocodigo])
pd.DataFrame({
    "m": np.arange(M, dtype=np.int16), "geocodigo": muns.geocodigo.astype(np.int32), "nome": muns.nome,
    "pop": np.round(POP[-1]).astype(np.int64),
    "limiar_pre": f32(LIM.limiar_pre), "limiar_pos": f32(LIM.limiar_pos), "limiar_epidemico": f32(LIM.limiar_epidemico),
    "limiar_proprio": LIM.limiar_proprio.to_numpy(), "lat": f32(centro[:, 0]), "lon": f32(centro[:, 1]),
}).to_parquet(DIR_PAINEL / "municipios.parquet", index=False)

feicoes = [{"type": "Feature", "id": int(g), "properties": {"nome": n},
            "geometry": mapping(shapely.set_precision(geoms[g].simplify(0.003, preserve_topology=True), 1e-4))}
           for g, n in zip(muns.geocodigo, muns.nome)]
(DIR_PAINEL / "malha.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": feicoes},
                                                     separators=(",", ":")), encoding="utf-8")

# 2. Séries semanais por município: casos, incidência, Rt e clima
pd.DataFrame({
    "semana": np.repeat(semanas.values, M), "m": np.tile(np.arange(M, dtype=np.int16), T),
    "casos": CASOS.ravel().astype(np.int32), "inc": f32(INC.ravel()),
    "rt": f32(np.clip(RT, 0, 10).ravel()), "p_rt": f32(P1.ravel()),
    "temp_min": f32(CLIMA["temp_min"].ravel()), "umid_med": f32(CLIMA["umid_med"].ravel()),
}).to_parquet(DIR_PAINEL / "serie.parquet", index=False, compression="zstd")

# 3. Previsões do período de teste (incidência por 100 mil hab.)
NIVEIS_Q = [0.05, 0.25, 0.5, 0.75, 0.95]
GRADE = np.linspace(0.005, 0.995, 100)
MULT = {}          # erros relativos do ensemble no teste: (real + c) / (previsto + c)


def prob_epidemia(prev, lim, mult):
    """P(incidência > limiar epidêmico) pela distribuição empírica dos erros relativos do ensemble."""
    return ((prev[:, None] + C_INC) * mult[None, :] - C_INC > lim[:, None]).mean(axis=1)


partes = []
for h in HORIZONTES:
    d = RES[h]
    MULT[h] = np.quantile((d.y.to_numpy() + C_INC) / (d.Ensemble.to_numpy() + C_INC), GRADE)
    mm = d.m.to_numpy()
    partes.append(pd.DataFrame({
        "h": np.int8(h), "semana_alvo": d.semana_alvo.to_numpy(), "m": mm.astype(np.int16),
        "ano_teste": d.ano_teste.to_numpy().astype(np.int16), "pop": f32(d.pop_alvo), "y": f32(d.y),
        "ensemble": f32(d.Ensemble), "rf": f32(d["RF (crescimento)"]), "linear": f32(d["Regressão linear"]),
        "persistencia": f32(d["Persistência"]),
        **{f"q{int(q * 100):02d}": f32(d[f"q{int(q * 100):02d}"]) for q in NIVEIS_Q},
        "p_epidemia": f32(prob_epidemia(d.Ensemble.to_numpy(), LIM_EPI_M[mm], MULT[h])),
        "escore_logistica": f32(CLF[h]["escores"]["Regressão logística"]),
        "escore_gb": f32(CLF[h]["escores"]["Gradient Boosting (classif.)"]),
    }))
pd.concat(partes, ignore_index=True).to_parquet(DIR_PAINEL / "previsoes_teste.parquet", index=False, compression="zstd")

# 4. Previsão a partir da última semana da base (modelos do último ano de teste)
futuro = []
for h in HORIZONTES:
    df = montar_painel(h)
    linha = df.iloc[(T - 1) * M: T * M].copy()
    alvo = semanas[-1] + pd.Timedelta(weeks=h)
    se_alvo = ano_se([alvo])[1][0]
    linha["se_sin"], linha["se_cos"] = np.sin(2 * np.pi * se_alvo / 52.18), np.cos(2 * np.pi * se_alvo / 52.18)
    ref = PAINEIS[h].semana_alvo <= TREINO_ATE
    for c in [c for c in FEATS if c.startswith(("temp", "umid"))]:
        linha[c] = linha[c].fillna(PAINEIS[h].loc[ref, c].mean())
    modelos, base = ULTIMOS_MODELOS[h][0], linha.inc_recente_media4.to_numpy()
    prev = [np.clip(modelos[n].predict(linha[FEATS]), 0, None) if "nível" in n
            else de_razao(modelos[n].predict(linha[FEATS]), base) for n in MODELOS_PONTO]
    ens = np.mean(prev, axis=0)
    qmult = np.quantile(MULT[h], NIVEIS_Q)
    futuro.append(pd.DataFrame({
        "h": np.int8(h), "semana_base": semanas[-1], "semana_alvo": alvo, "m": np.arange(M, dtype=np.int16),
        "ensemble": f32(ens),
        **{f"q{int(q * 100):02d}": f32(np.clip((ens + C_INC) * k - C_INC, 0, None)) for q, k in zip(NIVEIS_Q, qmult)},
        "p_epidemia": f32(prob_epidemia(ens, LIM_EPI_M, MULT[h])),
    }))
pd.concat(futuro, ignore_index=True).to_parquet(DIR_PAINEL / "previsao_futura.parquet", index=False)

# 5. Métricas e informações gerais
def tabela_json(tab):
    return json.loads(tab.round(4).reset_index(names="modelo").to_json(orient="records"))


meta = {
    "gerado_em": f"{pd.Timestamp.now():%d/%m/%Y %H:%M}", "uf": UF_SIGLA,
    "periodo": [str(semanas[0].date()), str(semanas[-1].date())], "n_municipios": int(M),
    "n_casos": int(CASOS.sum()), "horizontes": HORIZONTES, "anos_teste": ANOS_TESTE,
    "limiares_treino_ate": str(TREINO_ATE.date()),
    "regressao": {str(h): tabela_json(metricas_regressao(RES[h])) for h in HORIZONTES},
    "cobertura": {str(h): {k: round(float(v), 4) for k, v in cobertura(RES[h]).items()} for h in HORIZONTES},
    "classificacao": {str(h): tabela_json(pd.DataFrame({
        nome: metricas_classif(CLF[h]["y"], s, CLF[h]["cortes"][nome]) for nome, s in CLF[h]["escores"].items()}).T)
        for h in HORIZONTES},
}
(DIR_PAINEL / "metricas.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")

for f in sorted(DIR_PAINEL.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")
arq_zip = shutil.make_archive(str(DIR_PAINEL), "zip", DIR_PAINEL)
print(f"\nArquivos do painel em {DIR_PAINEL} e compactados em {arq_zip}")
if NO_COLAB:
    try:
        from google.colab import files
        files.download(arq_zip)
    except Exception as e:
        print(f"Download automático não funcionou ({e}); baixe o .zip pela pasta do Drive.")

## Referências

1. BRASIL. Ministério da Saúde. **Portal de Dados Abertos do SUS — Sinan/Dengue**. Disponível em: <https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue>.
2. BRASIL. Ministério da Saúde. **Dengue**. Disponível em: <https://www.gov.br/saude/pt-br/assuntos/saude-de-a-a-z/d/dengue>.
3. CODEÇO, C. T. et al. Infodengue: a nowcasting system for the surveillance of arboviruses in Brazil. *Revue d'Épidémiologie et de Santé Publique*, v. 66, p. S386, 2018.
4. CORI, A. et al. A new framework and software to estimate time-varying reproduction numbers during epidemics. *American Journal of Epidemiology*, v. 178, n. 9, p. 1505–1512, 2013.
5. IBGE. **SIDRA — tabelas 6579 e 4709**; **API de malhas territoriais**. Disponível em: <https://sidra.ibge.gov.br> e <https://servicodados.ibge.gov.br/api/docs/malhas?versao=3>.
6. INFODENGUE. **Informações e metodologia**. Disponível em: <https://info.dengue.mat.br/informacoes/>.
7. MUÑOZ-SABATER, J. et al. ERA5-Land: a state-of-the-art global reanalysis dataset for land applications. *Earth System Science Data*, v. 13, p. 4349–4383, 2021.
8. GOOGLE EARTH ENGINE. **ERA5-Land Daily Aggregates — ECMWF Climate Reanalysis** (`ECMWF/ERA5_LAND/DAILY_AGGR`). Disponível em: <https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR>.
9. GORELICK, N. et al. Google Earth Engine: planetary-scale geospatial analysis for everyone. *Remote Sensing of Environment*, v. 202, p. 18–27, 2017.
10. VEGA, T. et al. Influenza surveillance in Europe: establishing epidemic thresholds by the Moving Epidemic Method. *Influenza and Other Respiratory Viruses*, v. 7, n. 4, p. 546–558, 2013.
11. BRACHER, J. et al. Evaluating epidemic forecasts in an interval format. *PLOS Computational Biology*, v. 17, n. 2, e1008618, 2021.
12. CHEN, X.; MORAGA, P. Forecasting dengue across Brazil with LSTM neural networks and SHAP-driven lagged climate and spatial effects. *BMC Public Health*, 2025.
13. INFODENGUE-MOSQLIMATE. Leveraging probabilistic forecasts for dengue preparedness and control: the 2024 Dengue Forecasting Sprint in Brazil. *medRxiv*, 2025. DOI: 10.1101/2025.05.12.25327419.
14. PEDREGOSA, F. et al. Scikit-learn: machine learning in Python. *Journal of Machine Learning Research*, v. 12, p. 2825–2830, 2011.